SPDX-FileCopyrightText: 2026 Rodrigo López Guerra

SPDX-FileCopyrightText: 2026 Abigail Pérez García

SPDX-FileCopyrightText: 2026 Fernando Maggi Llerandi

SPDX-License-Identifier: Apache-2.0

# Generador de clones JavaScript con Vertex AI Batch

Objetivo:
- Generar clones Type I, II, III y IV.
- Evitar clones incompletos/truncados.
- Validar localmente sintaxis con `node --check`.
- Validar Type I por token-equivalence local.
- No usar ejecución con `stdin` vacío como filtro duro.
- Guardar aceptados, rechazados, reportes CSV y JSON.
- Permitir regenerar solo los fallidos.

In [1]:
# Cell 1 — Configuración

import json, random, re, csv, time, shutil, subprocess
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import pandas as pd
import vertexai
from vertexai.batch_prediction import BatchPredictionJob
from google.cloud import storage

# =========================
# Google Cloud
# =========================

PROJECT_ID    = "PROJECT-NAME"
LOCATION      = "us-central1"
BUCKET_NAME   = "BUCKET-NAME"
MODEL         = "gemini-2.5-flash"

# =========================
# Dataset
# =========================

DATASETS = {
    "dataset solutions": {
        "js_files": "./dataset/original",
        "clones": "./dataset/clones",
    },
    # "dataset general": {
    #     "js_files": "./dataset general/js_files",
    #     "clones": "./dataset general/clones",
    # },
}

# Qué tipos generar
CLONE_TYPES_TO_GENERATE = [1, 2, 3, 4]

# Si quieres regenerar solo algunos IDs, pon lista:
# ONLY_FILE_IDS = {"511", "372", "50"}
ONLY_FILE_IDS = None

# Si quieres saltar clones que ya existen y pasan validación local básica.
SKIP_EXISTING_VALID = False

# =========================
# Robustez / velocidad
# =========================

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# No recortar originales. Recortar fue una de las causas de falsos "incomplete".
MAX_LINES = None

# Mayor salida para evitar truncamiento.
# Si tu modelo/proyecto no soporta 16384, baja a 8192.
MAX_OUTPUT_TOKENS_BY_TYPE = {
    1: 8192,
    2: 16384,
    3: 16384,
    4: 16384,
}

# Temperaturas bajas para precisión. Type IV necesita algo más de flexibilidad.
TEMPERATURE_BY_TYPE = {
    1: 0.0,
    2: 0.1,
    3: 0.15,
    4: 0.2,
}

# Top P conservador.
TOP_P_BY_TYPE = {
    1: 0.95,
    2: 0.95,
    3: 0.95,
    4: 0.95,
}

# Si el output es demasiado corto contra el original, marcar como sospechoso.
# No es fallo duro si pasa node --check, pero se reporta.
SHORT_LINE_RATIO_WARNING = 0.35

# =========================
# Carpetas
# =========================

OUTPUT_FOLDER = Path("./generation")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

REPORT_DIR = OUTPUT_FOLDER / f"run_{RUN_ID}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_INPUT_DIR = REPORT_DIR / "batch_input"
BATCH_OUTPUT_DIR = REPORT_DIR / "batch_output"
REPORTS_DIR = REPORT_DIR / "reports"
REJECTED_DIR = REPORT_DIR / "rejected_outputs"

BATCH_INPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REJECTED_DIR.mkdir(parents=True, exist_ok=True)

vertexai.init(project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client(project=PROJECT_ID)

print("Config OK")
print("RUN_ID:", RUN_ID)
print("MODEL:", MODEL)
print("MAX_LINES:", MAX_LINES)
print("OUTPUT:", REPORT_DIR)

Config OK
RUN_ID: 20260629_031953
MODEL: gemini-2.5-flash
MAX_LINES: None
OUTPUT: generation/run_20260629_031953


In [2]:
# Cell 2 — Verificar Node y Google Cloud Storage

def get_node_command():
    """
    Detecta cómo ejecutar Node.
    Caso normal:
        node
    Caso VSCodium Flatpak:
        flatpak-spawn --host node
    """
    if shutil.which("node"):
        return ["node"]

    if shutil.which("flatpak-spawn"):
        test = subprocess.run(
            ["flatpak-spawn", "--host", "node", "--version"],
            capture_output=True,
            text=True
        )
        if test.returncode == 0:
            return ["flatpak-spawn", "--host", "node"]

    raise RuntimeError(
        "Node.js no está disponible. En VSCodium Flatpak verifica con: "
        "flatpak-spawn --host node --version"
    )

NODE_CMD = get_node_command()

node_version = subprocess.run(NODE_CMD + ["--version"], capture_output=True, text=True)
print("NODE_CMD:", NODE_CMD)
print("Node version:", node_version.stdout.strip())

def test_bucket_permissions():
    print("PROJECT_ID:", PROJECT_ID)
    print("BUCKET_NAME:", BUCKET_NAME)

    bucket = storage_client.bucket(BUCKET_NAME)

    if not bucket.exists():
        raise RuntimeError(f"El bucket no existe o no tienes acceso: gs://{BUCKET_NAME}")

    test_blob = bucket.blob(f"permission_test/{RUN_ID}.txt")

    try:
        test_blob.upload_from_string("ok", content_type="text/plain")
        test_blob.delete()
        print("GCS permission test: OK")
    except Exception as e:
        print("GCS permission test: FAILED")
        print(type(e).__name__)
        print(e)
        raise

test_bucket_permissions()

NODE_CMD: ['node']
Node version: v26.4.0
PROJECT_ID: PROJECT-NAME
BUCKET_NAME: BUCKET-NAME


BadRequest: 400 GET https://storage.googleapis.com/storage/v1/b/BUCKET-NAME?fields=name&prettyPrint=false: Invalid bucket name: 'BUCKET-NAME'

In [3]:
# Cell 3 — Prompts de generación

COMMON_OUTPUT_RULES = """
Universal output rules:
- Output only raw JavaScript code.
- No markdown.
- No backticks.
- No explanations.
- No headings.
- No comments outside the code.
- The output must be a complete JavaScript file.
- Do not stop in the middle of a statement, comment, string, block, function, loop, array, object, or conditional.
- All opened braces, brackets, parentheses, comments, strings, and template literals must be closed.
- The final answer must be syntactically valid JavaScript.
- If you cannot fit the complete clone in the output, respond exactly with:
GENERATION_TOO_LONG
"""

STRICT_BEHAVIOR_RULES = """
Strict behavioral equivalence:
- Preserve the exact observable behavior of the original for equivalent inputs.
- Preserve exact stdout, stderr, thrown errors, return values, side effects, async behavior, and external contracts.
- Console output is behavior-critical.
- Do not add labels, headings, debug text, explanations, prefixes, suffixes, or extra output lines.
- Do not change string contents used in output, comparisons, object keys, file paths, imports, exports, API calls, or user-visible messages.
- Do not change hard-coded sample data, numeric constants, array contents, object values, loop bounds, loop inclusivity, or iteration counts unless mathematically and observably identical.
"""

CLONE_PROMPTS = {
    1: f"""You are an expert in JavaScript Type I code clone generation.

Generate a Type I clone of the JavaScript code below.

Official Type I definition:
Code fragments are exactly equal, except for possible variations in whitespace and comments.

Hard Type I rules:
- Only whitespace and comments may change.
- Preserve the exact same sequence of non-comment lexical tokens.
- Do not rename identifiers.
- Do not change literals.
- Do not change operators.
- Do not add or remove semicolons, braces, parentheses, imports, exports, statements, or executable code.
- Do not rewrite expressions.
- Do not fix or modernize the code.
- Do not change quote style.
- Do not change declaration style.

Required changes:
- Reformat indentation and spacing.
- Add, remove, or rewrite comments.
- Add or remove blank lines where valid.
- Make it visually different without changing executable tokens.

{STRICT_BEHAVIOR_RULES}

Before output:
- Verify that removing comments and whitespace gives the same token sequence as the original.
- Verify the output is a complete valid JavaScript file.

{COMMON_OUTPUT_RULES}""",

    2: f"""You are an expert in JavaScript Type II code clone generation.

Generate a Type II clone of the JavaScript code below.

Official Type II definition:
The structure is the same, but identifier names, data types where applicable, whitespace and comments differ.

Hard Type II rules:
- Preserve the same control flow, statement order, operators, expressions, algorithm, and behavior.
- Do not add meaningful executable logic.
- Do not delete executable logic.
- Do not convert the algorithm to a different strategy.
- Rename user-defined identifiers consistently across the whole file.
- Do not rename JavaScript built-ins, standard globals, library APIs, DOM APIs, imported package names, external contracts, or behavior-critical property names.
- Object properties may be renamed only if all reads/writes in the file are updated consistently and behavior is unchanged.
- Keep the implementation recognizably Type II, not Type III or Type IV.

Allowed surface changes:
- Whitespace and comments.
- User-defined identifier names.
- Harmless literal representations only when runtime value and observable behavior are identical.

{STRICT_BEHAVIOR_RULES}

Before output:
- Verify the clone is complete and syntactically valid JavaScript.
- Verify the same structure and behavior are preserved.
- Verify user-defined identifiers are consistently renamed.

{COMMON_OUTPUT_RULES}""",

    3: f"""You are an expert in JavaScript Type III code clone generation.

Generate a Type III clone of the JavaScript code below.

Official Type III definition:
Like Type II, but with moderate additions, deletions, or modifications of code fragments.

Hard Type III rules:
- Preserve exact observable behavior.
- Apply Type II-style identifier renaming first where safe.
- Add moderate structural edits beyond simple renaming/reformatting.
- Keep the result recognizably related to the original.
- Do not turn it into a completely different Type IV strategy.
- Do not remove essential logic, outputs, validation, returns, side effects, imports, exports, or behavior-critical statements.
- Added code must be meaningful, not dead or decorative.
- Removed code must be redundant, cosmetic, or non-essential.

Good Type III edits:
- helper variables
- temporary constants
- guard clauses
- small helper functions
- splitting or merging simple statements
- rewriting local expressions equivalently
- adding intermediate computation steps

Forbidden edits:
- changing output text or formatting
- changing hard-coded data
- changing loop ranges or iteration counts
- adding user-facing messages
- deleting main logic

{STRICT_BEHAVIOR_RULES}

Before output:
- Verify the clone is complete and syntactically valid JavaScript.
- Verify the behavior is externally indistinguishable.
- Verify it is more than Type II but not Type IV.

{COMMON_OUTPUT_RULES}""",

    4: f"""You are an expert in JavaScript Type IV code clone generation.

Generate a Type IV clone of the JavaScript code below.

Official Type IV definition:
The fragments use divergent syntax but execute equivalent functionality.

Hard Type IV rules:
- Preserve exact observable behavior and external contract.
- Use a substantially different implementation strategy in the core logic.
- Do not merely rename variables or reformat code.
- Do not keep the same statement structure or control-flow structure unless unavoidable.
- Do not reuse original user-defined identifier names unless required by imports, exports, external APIs, or runtime behavior.
- Do not rename JavaScript built-ins, standard globals, library APIs, DOM APIs, imported package names, or behavior-critical external names.

Good Type IV strategies:
- loops ↔ functional methods / reducers / recursion / generators
- conditionals ↔ lookup tables / dispatch maps / expression-based logic
- imperative ↔ declarative or functional style
- different helper decomposition
- different data representation
- different core expression design

Forbidden:
- changing output text
- changing constants or sample data
- adding debug/explanation output
- changing loop ranges
- changing external APIs or exports

{STRICT_BEHAVIOR_RULES}

Before output:
- Verify the clone is complete and syntactically valid JavaScript.
- Verify exact external behavior.
- Verify the core implementation strategy is substantially different.

{COMMON_OUTPUT_RULES}""",
}

print("Prompts OK")

Prompts OK


In [4]:
# Cell 4 — Funciones locales robustas

def read_code(path):
    return Path(path).read_text(encoding="utf-8", errors="ignore")

def limit_code_text(text):
    if MAX_LINES is None:
        return text
    lines = text.splitlines()
    if len(lines) <= MAX_LINES:
        return text
    return "\n".join(lines[:MAX_LINES]) + "\n\n/* [TRUNCATED_FOR_GENERATION] */"

def clean_model_code(text):
    if text is None:
        return ""
    text = text.strip()

    if text == "GENERATION_TOO_LONG":
        return text

    # Quitar fences si el modelo los puso.
    if text.startswith("```"):
        text = re.sub(r"^```(?:javascript|js)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    # Evitar textos accidentales antes/después: conservador, no intenta parsear agresivamente.
    return text.strip() + "\n"

def node_syntax_check(path, timeout=8):
    try:
        result = subprocess.run(
            NODE_CMD + ["--check", str(Path(path).resolve())],
            capture_output=True,
            text=True,
            timeout=timeout
        )
        return result.returncode == 0, result.stderr.strip()
    except subprocess.TimeoutExpired:
        return False, "timeout"
    except Exception as e:
        return False, str(e)

def syntax_check_text(code_text, temp_path=None):
    if temp_path is None:
        temp_path = REPORT_DIR / "_syntax_temp.js"
    temp_path = Path(temp_path)
    temp_path.write_text(code_text, encoding="utf-8")
    return node_syntax_check(temp_path)

def normalize_js_for_type1(code):
    """
    Quita whitespace y comentarios fuera de strings/templates.
    """
    out = []
    i = 0
    n = len(code)
    state = "normal"
    quote = None

    while i < n:
        ch = code[i]
        nxt = code[i + 1] if i + 1 < n else ""

        if state == "normal":
            if ch.isspace():
                i += 1
                continue
            if ch == "/" and nxt == "/":
                i += 2
                while i < n and code[i] not in "\r\n":
                    i += 1
                continue
            if ch == "/" and nxt == "*":
                i += 2
                while i + 1 < n and not (code[i] == "*" and code[i + 1] == "/"):
                    i += 1
                i += 2
                continue
            if ch in ("'", '"', "`"):
                state = "string"
                quote = ch
                out.append(ch)
                i += 1
                continue
            out.append(ch)
            i += 1
            continue

        if state == "string":
            out.append(ch)
            if ch == "\\" and i + 1 < n:
                out.append(code[i + 1])
                i += 2
                continue
            if ch == quote:
                state = "normal"
                quote = None
            i += 1

    return "".join(out)

def type1_tokens_equivalent(original_code, clone_code):
    return normalize_js_for_type1(original_code) == normalize_js_for_type1(clone_code)

def has_common_output_call(text):
    patterns = [
        r"\bconsole\.log\s*\(",
        r"\bconsole\.error\s*\(",
        r"\bprocess\.stdout\.write\s*\(",
        r"\bprocess\.stderr\.write\s*\(",
        r"\bfs\.writeSync\s*\(\s*1\s*,",
        r"\bfs\.writeSync\s*\(\s*2\s*,",
    ]
    return any(re.search(p, text) for p in patterns)

def uses_stdin(text):
    patterns = [
        r"fs\.readFileSync\s*\(\s*0",
        r"fs\.readFileSync\s*\(\s*['\"]\/dev\/stdin['\"]",
        r"process\.stdin",
        r"readline\.createInterface",
        r"\brl\.on\s*\(",
        r"\.on\s*\(\s*['\"]line['\"]",
        r"\.on\s*\(\s*['\"]close['\"]",
    ]
    return any(re.search(p, text) for p in patterns)

def suspicious_short_clone(original_code, clone_code):
    orig_lines = max(1, len(original_code.splitlines()))
    clone_lines = len(clone_code.splitlines())
    ratio = clone_lines / orig_lines
    return ratio < SHORT_LINE_RATIO_WARNING, ratio

def basic_original_ok(path):
    path = Path(path)
    if not path.exists():
        return False, "missing_original"
    text = read_code(path)
    if not text.strip():
        return False, "empty_original"
    return True, ""

def existing_clone_valid(original_path, clone_path, clone_type):
    if not Path(clone_path).exists():
        return False
    code_text = read_code(clone_path)
    if not code_text.strip():
        return False
    syntax_ok, _ = node_syntax_check(clone_path)
    if not syntax_ok:
        return False
    if clone_type == 1:
        return type1_tokens_equivalent(read_code(original_path), code_text)
    return True

print("Funciones locales OK")

Funciones locales OK


In [5]:
# Cell 5 — Preparar requests de generación

requests = []
meta = []
skipped_existing = []
local_rejects = []
prep_stats = defaultdict(int)

for dataset_name, paths in DATASETS.items():
    jsfiles_dir = Path(paths["js_files"])
    clones_dir = Path(paths["clones"])
    clones_dir.mkdir(parents=True, exist_ok=True)

    originals = sorted(jsfiles_dir.glob("*.js"))
    print(f"{dataset_name}: {len(originals)} originales encontrados")

    for orig_path in originals:
        file_id = orig_path.stem

        if ONLY_FILE_IDS is not None and file_id not in set(map(str, ONLY_FILE_IDS)):
            continue

        ok, reason = basic_original_ok(orig_path)
        if not ok:
            prep_stats[f"{dataset_name}_bad_original"] += 1
            local_rejects.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": None,
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}",
                "status": "bad_original",
                "reason": reason,
                "original_path": str(orig_path),
            })
            continue

        original_code_raw = read_code(orig_path)
        original_code_for_prompt = limit_code_text(original_code_raw)

        if "[TRUNCATED_FOR_GENERATION]" in original_code_for_prompt:
            local_rejects.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": None,
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}",
                "status": "original_too_long",
                "reason": "Original was truncated for generation. Set MAX_LINES=None or handle separately.",
                "original_path": str(orig_path),
            })
            continue

        for clone_type in CLONE_TYPES_TO_GENERATE:
            clone_name = f"{file_id}_clon_{clone_type}.js"
            clone_path = clones_dir / clone_name
            custom_id = f"{dataset_name.replace(' ', '_')}|{clone_name}"

            if SKIP_EXISTING_VALID and existing_clone_valid(orig_path, clone_path, clone_type):
                skipped_existing.append({
                    "id": custom_id,
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "clone_type": clone_type,
                    "clone_path": str(clone_path),
                    "status": "skipped_existing_valid",
                })
                prep_stats["skipped_existing_valid"] += 1
                continue

            prompt_text = (
                CLONE_PROMPTS[clone_type]
                + f"\n\nPair ID: {custom_id}"
                + f"\nDeclared clone type: {clone_type}"
                + f"\nOriginal line count: {len(original_code_raw.splitlines())}"
                + f"\nOriginal character count: {len(original_code_raw)}"
                + f"\nOriginal uses stdin: {uses_stdin(original_code_raw)}"
                + f"\nOriginal has common output call: {has_common_output_call(original_code_raw)}"
                + "\n\n[ORIGINAL_COMPLETE_FILE_START]\n"
                + original_code_for_prompt
                + "\n[ORIGINAL_COMPLETE_FILE_END]"
            )

            req = {
                "custom_id": custom_id,
                "request": {
                    "model": f"models/{MODEL}",
                    "contents": [
                        {
                            "role": "user",
                            "parts": [{"text": prompt_text}],
                        }
                    ],
                    "generation_config": {
                        "temperature": TEMPERATURE_BY_TYPE.get(clone_type, 0.1),
                        "top_p": TOP_P_BY_TYPE.get(clone_type, 0.95),
                        "max_output_tokens": MAX_OUTPUT_TOKENS_BY_TYPE.get(clone_type, 16384),
                    },
                },
            }

            requests.append(req)
            meta.append({
                "id": custom_id,
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "original_lines": len(original_code_raw.splitlines()),
                "original_chars": len(original_code_raw),
                "temperature": TEMPERATURE_BY_TYPE.get(clone_type, 0.1),
                "max_output_tokens": MAX_OUTPUT_TOKENS_BY_TYPE.get(clone_type, 16384),
            })
            prep_stats["requests"] += 1

jsonl_path = BATCH_INPUT_DIR / "generation_requests.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for req in requests:
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

with open(REPORTS_DIR / "generation_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

pd.DataFrame(skipped_existing).to_csv(REPORTS_DIR / "skipped_existing_valid.csv", index=False)
pd.DataFrame(local_rejects).to_csv(REPORTS_DIR / "local_rejects_before_generation.csv", index=False)

print("Preparación terminada")
print("Requests:", len(requests))
print("Skipped existing valid:", len(skipped_existing))
print("Local rejects before generation:", len(local_rejects))
print("JSONL:", jsonl_path)
print(dict(prep_stats))

dataset solutions: 11001 originales encontrados
Preparación terminada
Requests: 44004
Skipped existing valid: 0
Local rejects before generation: 0
JSONL: generation/run_20260629_031953/batch_input/generation_requests.jsonl
{'requests': 44004}


In [ ]:
# Cell 6 — Subir JSONL y lanzar batch con timeout largo

from pathlib import Path
import json
import time

def upload_jsonl_to_gcs(jsonl_path, subdir="generation"):
    jsonl_path = Path(jsonl_path)
    size_mb = jsonl_path.stat().st_size / (1024 * 1024)

    print("=" * 80)
    print("UPLOAD JSONL")
    print("=" * 80)
    print("Local file:", jsonl_path)
    print(f"Size: {size_mb:.2f} MB")
    print("Requests:", len(requests))

    bucket = storage_client.bucket(BUCKET_NAME)
    blob_name = f"{subdir}/input/run_{RUN_ID}/{jsonl_path.name}"
    blob = bucket.blob(blob_name)

    # Chunk grande para evitar demasiadas partes.
    # Debe ser múltiplo de 256 KB.
    blob.chunk_size = 32 * 1024 * 1024

    start = time.time()
    print("Uploading to GCS...")

    blob.upload_from_filename(
        str(jsonl_path),
        content_type="application/jsonl",
        timeout=3600,
    )

    elapsed = time.time() - start
    gcs_uri = f"gs://{BUCKET_NAME}/{blob_name}"

    print("Upload OK")
    print("GCS URI:", gcs_uri)
    print(f"Upload time: {elapsed/60:.2f} min")

    return gcs_uri


def submit_generation_batch(jsonl_path):
    input_uri = upload_jsonl_to_gcs(jsonl_path, subdir="generation")
    output_uri = f"gs://{BUCKET_NAME}/generation/output/run_{RUN_ID}/"

    print("\n" + "=" * 80)
    print("SUBMIT BATCH")
    print("=" * 80)
    print("Input :", input_uri)
    print("Output:", output_uri)
    print("Submitting generation batch...")

    start = time.time()

    job = BatchPredictionJob.submit(
        source_model=MODEL,
        input_dataset=input_uri,
        output_uri_prefix=output_uri,
    )

    elapsed = time.time() - start

    print("\nBatch submitted")
    print("Job name:", job.resource_name)
    print("State:", job.state)
    print(f"Submit time: {elapsed:.2f} sec")

    return job.resource_name, output_uri


if len(requests) == 0:
    print("No hay requests. No se lanza batch.")
    job_name = None
    output_uri = None
else:
    job_name, output_uri = submit_generation_batch(jsonl_path)

batch_info = {
    "job_name": job_name,
    "output_uri": output_uri,
    "run_id": RUN_ID,
    "jsonl_path": str(jsonl_path),
    "num_requests": len(requests),
}

with open(REPORTS_DIR / "batch_info.json", "w", encoding="utf-8") as f:
    json.dump(batch_info, f, ensure_ascii=False, indent=2)

batch_info

In [ ]:
# Cell 7 — Revisar estado del batch

def refresh_job(job_name):
    if not job_name:
        print("No hay job_name.")
        return None
    job = BatchPredictionJob(job_name)
    job.refresh()
    print("Job:", job.resource_name)
    print("State:", job.state)
    return job

try:
    job_name
except NameError:
    with open(REPORTS_DIR / "batch_info.json", "r", encoding="utf-8") as f:
        batch_info = json.load(f)
    job_name = batch_info["job_name"]
    output_uri = batch_info["output_uri"]

job = refresh_job(job_name)

In [ ]:
# Cell 8 — Descargar, parsear, guardar clones y validar localmente

def extract_text_from_batch_line(data):
    custom_id = data.get("custom_id") or data.get("id") or "unknown"

    if "error" in data:
        return custom_id, None, f"model_error: {data['error']}"

    try:
        text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
        return custom_id, text, ""
    except Exception as e:
        return custom_id, None, f"parse_error: {e}"

def download_batch_jsonl(output_uri):
    bucket = storage_client.bucket(BUCKET_NAME)
    prefix = output_uri.replace(f"gs://{BUCKET_NAME}/", "")
    blobs = list(bucket.list_blobs(prefix=prefix))

    lines = []
    print("Blobs encontrados:", len(blobs))

    for blob in blobs:
        if not blob.name.endswith(".jsonl"):
            continue
        print("Leyendo:", blob.name)
        content = blob.download_as_text()
        for line in content.splitlines():
            if line.strip():
                lines.append(line)

    return lines

try:
    output_uri
except NameError:
    with open(REPORTS_DIR / "batch_info.json", "r", encoding="utf-8") as f:
        batch_info = json.load(f)
    output_uri = batch_info["output_uri"]

meta_path = REPORTS_DIR / "generation_meta.json"
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

meta_by_id = {m["id"]: m for m in meta}

raw_lines = download_batch_jsonl(output_uri)

results = []
accepted = []
rejected = []

for raw in raw_lines:
    try:
        data = json.loads(raw)
    except Exception as e:
        rejected.append({
            "id": "unknown",
            "status": "line_parse_error",
            "reason": str(e),
        })
        continue

    custom_id, raw_text, model_error = extract_text_from_batch_line(data)
    m = meta_by_id.get(custom_id, {})

    dataset_name = m.get("dataset", "")
    file_id = str(m.get("file_id", ""))
    clone_type = int(m.get("clone_type", 0)) if m.get("clone_type") is not None else None

    original_path = Path(m.get("original_path", "")) if m.get("original_path") else None
    clone_path = Path(m.get("clone_path", "")) if m.get("clone_path") else None

    row = {
        "id": custom_id,
        "dataset": dataset_name,
        "file_id": file_id,
        "clone_type": clone_type,
        "original_path": str(original_path) if original_path else "",
        "clone_path": str(clone_path) if clone_path else "",
        "status": "",
        "reason": "",
        "syntax_ok": None,
        "type1_token_ok": None,
        "short_line_warning": False,
        "line_ratio": None,
        "raw_chars": len(raw_text) if raw_text else 0,
    }

    if model_error:
        row["status"] = "rejected_model_error"
        row["reason"] = model_error
        rejected.append(row)
        continue

    code_text = clean_model_code(raw_text)

    if code_text.strip() == "GENERATION_TOO_LONG":
        row["status"] = "rejected_generation_too_long"
        row["reason"] = "Model reported GENERATION_TOO_LONG"
        rejected.append(row)
        continue

    if not code_text.strip():
        row["status"] = "rejected_empty_output"
        row["reason"] = "Empty model output"
        rejected.append(row)
        continue

    original_code = read_code(original_path) if original_path and original_path.exists() else ""

    # Validación de sintaxis sobre texto antes de guardar definitivo
    temp_path = REPORT_DIR / f"_syntax_{file_id}_clon_{clone_type}.js"
    syntax_ok, syntax_error = syntax_check_text(code_text, temp_path=temp_path)

    row["syntax_ok"] = syntax_ok

    if not syntax_ok:
        row["status"] = "rejected_syntax_error"
        row["reason"] = syntax_error[:500]
        rejected_output_path = REJECTED_DIR / f"{file_id}_clon_{clone_type}.js"
        rejected_output_path.write_text(code_text, encoding="utf-8")clon_
        row["rejected_output_path"] = str(rejected_output_path)
        rejected.append(row)
        continue

    # Type I local
    if clone_type == 1:
        token_ok = type1_tokens_equivalent(original_code, code_text)
        row["type1_token_ok"] = token_ok
        if not token_ok:
            row["status"] = "rejected_type1_token_mismatch"
            row["reason"] = "Type I token-equivalence failed"
            rejected_output_path = REJECTED_DIR / f"{file_id}_clon_{clone_type}.js"
            rejected_output_path.write_text(code_text, encoding="utf-8")
            row["rejected_output_path"] = str(rejected_output_path)
            rejected.append(row)
            continue

    short_warn, ratio = suspicious_short_clone(original_code, code_text)
    row["short_line_warning"] = bool(short_warn)
    row["line_ratio"] = ratio

    # Guardar clon aceptado
    clone_path.parent.mkdir(parents=True, exist_ok=True)
    clone_path.write_text(code_text, encoding="utf-8")

    row["status"] = "accepted"
    row["reason"] = "Syntax OK" + ("; short_line_warning" if short_warn else "")
    accepted.append(row)
    results.append(row)

# Guardar reportes
all_rows = accepted + rejected

df_all = pd.DataFrame(all_rows)
df_accepted = pd.DataFrame(accepted)
df_rejected = pd.DataFrame(rejected)

df_all.to_csv(REPORTS_DIR / "generation_results_all.csv", index=False)
df_accepted.to_csv(REPORTS_DIR / "generation_accepted.csv", index=False)
df_rejected.to_csv(REPORTS_DIR / "generation_rejected.csv", index=False)

with open(REPORTS_DIR / "generation_results_all.json", "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)

print("Parseo terminado")
print("Raw lines:", len(raw_lines))
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

if len(df_rejected):
    display(df_rejected["status"].value_counts())
else:
    print("No rejected outputs.")

if len(df_all):
    display(pd.crosstab(df_all["clone_type"], df_all["status"], margins=True))

Blobs encontrados: 1
Leyendo: generation/output/run_20260517_162122/prediction-model-2026-05-17T22:24:34.213222Z/predictions.jsonl
Parseo terminado
Raw lines: 20044
Accepted: 16984
Rejected: 3060


status
rejected_type1_token_mismatch    2159
rejected_syntax_error             785
line_parse_error                   72
rejected_generation_too_long       40
rejected_empty_output               4
Name: count, dtype: int64

status,accepted,rejected_empty_output,rejected_generation_too_long,rejected_syntax_error,rejected_type1_token_mismatch,All
clone_type,,,,,,
1.0,2414,2,26,392,2159,4993
2.0,4826,2,5,160,0,4993
3.0,4888,0,8,97,0,4993
4.0,4856,0,1,136,0,4993
All,16984,4,40,785,2159,19972


In [26]:
# Cell 8.5 — Configuración especial para retry

# El retry NO debe tocar Type I.
# Type I se regenera localmente, no con LLM.
# Retry LLM solo se usa para Type II / III / IV rechazados.

RETRY_TEMPERATURE_BY_TYPE = {
    1: 0.0,
    2: 0.05,
    3: 0.10,
    4: 0.15,
}

RETRY_TOP_P_BY_TYPE = {
    1: 0.90,
    2: 0.90,
    3: 0.90,
    4: 0.90,
}

RETRY_MAX_OUTPUT_TOKENS_BY_TYPE = {
    1: 8192,
    2: 16384,
    3: 16384,
    4: 16384,
}

print("Retry config OK")
print("RETRY_TEMPERATURE_BY_TYPE:", RETRY_TEMPERATURE_BY_TYPE)
print("RETRY_TOP_P_BY_TYPE:", RETRY_TOP_P_BY_TYPE)
print("RETRY_MAX_OUTPUT_TOKENS_BY_TYPE:", RETRY_MAX_OUTPUT_TOKENS_BY_TYPE)

Retry config OK
RETRY_TEMPERATURE_BY_TYPE: {1: 0.0, 2: 0.05, 3: 0.1, 4: 0.15}
RETRY_TOP_P_BY_TYPE: {1: 0.9, 2: 0.9, 3: 0.9, 4: 0.9}
RETRY_MAX_OUTPUT_TOKENS_BY_TYPE: {1: 8192, 2: 16384, 3: 16384, 4: 16384}


In [15]:
# Cell 8.6 — Regenerar Type I localmente, sin LLM, con comentarios multilingües ricos

# Reglas Type I:
# - Solo se puede cambiar whitespace y comentarios.
# - No se debe tocar ningún token ejecutable.
# - Cada archivo usa UNA sola lengua para todos sus comentarios.
# - No se menciona que es un clon.
# - Puede o no aparecer un encabezado de proyecto.
# - Si aparece nombre de proyecto, debe ser único por archivo.
# - Si aparecen integrantes, deben ser de 1 a 5 nombres únicos.
# - Puede haber secciones/separadores, pero no siempre.
# - Se valida con:
#   1. type1_tokens_equivalent
#   2. node --check

import random
import hashlib
import itertools
import pandas as pd
from pathlib import Path

TYPE1_LOCAL_SEED = 42
MIN_VARIANTS_PER_ELEMENT = 100
MIN_PHRASES_PER_LANGUAGE = 20000

TYPE1_LANGUAGES = [
    "fr",  # francés
    "de",  # alemán
    "es",  # español
    "en",  # inglés
    "ko",  # coreano
    "ja",  # japonés
    "hi",  # hindi
    "zh",  # chino
    "pt",  # portugués
    "ru",  # ruso
    "la",  # latín
]

LANGUAGE_NAMES = {
    "fr": "francés",
    "de": "alemán",
    "es": "español",
    "en": "inglés",
    "ko": "coreano",
    "ja": "japonés",
    "hi": "hindi",
    "zh": "chino",
    "pt": "portugués",
    "ru": "ruso",
    "la": "latín",
}

# ============================================================
# Bases por idioma
# ============================================================

LANG_BASE = {
    "es": {
        "subjects": [
            "la lectura", "el flujo", "la entrada", "la salida", "el cálculo", "la condición",
            "la iteración", "la función", "el bloque", "la estructura", "la validación",
            "el arreglo", "el objeto", "la cadena", "el índice", "el contador", "el resultado",
            "la conversión", "la comparación", "la rama",
        ],
        "actions": [
            "organiza", "mantiene", "prepara", "separa", "ordena", "describe", "acompaña",
            "delimita", "agrupa", "resalta", "aclara", "guía", "marca", "sostiene",
            "protege", "explica", "ubica", "resume", "refuerza", "presenta",
        ],
        "objects": [
            "los datos recibidos", "la ruta de procesamiento", "la información calculada",
            "el resultado visible", "la decisión local", "la secuencia de trabajo",
            "la forma del programa", "el valor intermedio", "el recorrido de datos",
            "la salida esperada", "la lectura humana", "la continuidad del código",
            "el análisis de valores", "la preparación de variables", "el cierre del bloque",
            "la operación principal", "el paso siguiente", "el contexto cercano",
            "la transformación aplicada", "el orden de evaluación",
        ],
        "qualifiers": [
            "sin modificar la ejecución", "sin cambiar los resultados", "como apoyo visual",
            "para facilitar revisión", "con intención descriptiva", "sin alterar valores",
            "en una zona segura", "para mejorar legibilidad", "sin afectar la salida",
            "en el mismo contexto", "con alcance local", "sin tocar operadores",
            "sin alterar literales", "como anotación auxiliar", "para separar ideas",
            "sin cambiar contratos", "manteniendo claridad", "para lectura rápida",
            "sin modificar decisiones", "con propósito documental",
        ],
        "project_words_a": ["Aurora", "Bruma", "Cauce", "Lumbre", "Niebla", "Risco", "Umbral", "Vega", "Cedro", "Nácar"],
        "project_words_b": ["Azul", "Sereno", "Claro", "Norte", "Dorado", "Firme", "Lunar", "Manso", "Vivo", "Alto"],
        "names": [
            "Sofía", "Mateo", "Valentina", "Santiago", "Camila", "Diego", "Lucía", "Emiliano",
            "Mariana", "Sebastián", "Isabella", "Nicolás", "Renata", "Julián", "Paula", "Andrés",
            "Elena", "Tomás", "Carolina", "Gabriel", "Martina", "Rafael", "Ana", "Héctor",
            "Clara", "Iván", "Adriana", "Luis", "Mónica", "Pablo",
        ],
        "header_project": "Proyecto",
        "header_team": "Equipo",
    },
    "en": {
        "subjects": [
            "the reading", "the flow", "the input", "the output", "the calculation", "the condition",
            "the iteration", "the function", "the block", "the structure", "the validation",
            "the array", "the object", "the string", "the index", "the counter", "the result",
            "the conversion", "the comparison", "the branch",
        ],
        "actions": [
            "organizes", "keeps", "prepares", "separates", "orders", "describes", "supports",
            "delimits", "groups", "highlights", "clarifies", "guides", "marks", "sustains",
            "protects", "explains", "places", "summarizes", "reinforces", "presents",
        ],
        "objects": [
            "the received data", "the processing route", "the computed information",
            "the visible result", "the local decision", "the working sequence",
            "the program shape", "the intermediate value", "the data traversal",
            "the expected output", "the human reading", "the code continuity",
            "the value analysis", "the variable preparation", "the block closure",
            "the main operation", "the next step", "the nearby context",
            "the applied transformation", "the evaluation order",
        ],
        "qualifiers": [
            "without changing execution", "without changing results", "as visual support",
            "to ease review", "with descriptive intent", "without altering values",
            "in a safe area", "to improve readability", "without affecting output",
            "in the same context", "with local scope", "without touching operators",
            "without altering literals", "as an auxiliary note", "to separate ideas",
            "without changing contracts", "while keeping clarity", "for quick reading",
            "without modifying decisions", "for documentation purposes",
        ],
        "project_words_a": ["Aurora", "Cedar", "Harbor", "Willow", "Quartz", "Meadow", "Summit", "Ember", "Silver", "Briar"],
        "project_words_b": ["North", "Clear", "Bright", "Quiet", "Steady", "Golden", "Lunar", "Green", "High", "Calm"],
        "names": [
            "Oliver", "Amelia", "Henry", "Charlotte", "James", "Evelyn", "Lucas", "Grace",
            "Benjamin", "Nora", "Samuel", "Lily", "Ethan", "Claire", "Daniel", "Alice",
            "Mason", "Ruby", "Leo", "Violet", "Nathan", "Ivy", "Adam", "Rose",
            "Owen", "Maya", "Isaac", "Eva", "Caleb", "Sarah",
        ],
        "header_project": "Project",
        "header_team": "Team",
    },
    "fr": {
        "subjects": [
            "la lecture", "le flux", "l'entrée", "la sortie", "le calcul", "la condition",
            "l'itération", "la fonction", "le bloc", "la structure", "la validation",
            "le tableau", "l'objet", "la chaîne", "l'indice", "le compteur", "le résultat",
            "la conversion", "la comparaison", "la branche",
        ],
        "actions": [
            "organise", "maintient", "prépare", "sépare", "ordonne", "décrit", "soutient",
            "délimite", "regroupe", "met en évidence", "clarifie", "guide", "marque", "porte",
            "protège", "explique", "place", "résume", "renforce", "présente",
        ],
        "objects": [
            "les données reçues", "le parcours de traitement", "l'information calculée",
            "le résultat visible", "la décision locale", "la séquence de travail",
            "la forme du programme", "la valeur intermédiaire", "le parcours des données",
            "la sortie attendue", "la lecture humaine", "la continuité du code",
            "l'analyse des valeurs", "la préparation des variables", "la fermeture du bloc",
            "l'opération principale", "l'étape suivante", "le contexte voisin",
            "la transformation appliquée", "l'ordre d'évaluation",
        ],
        "qualifiers": [
            "sans modifier l'exécution", "sans changer les résultats", "comme appui visuel",
            "pour faciliter la revue", "avec une intention descriptive", "sans altérer les valeurs",
            "dans une zone sûre", "pour améliorer la lisibilité", "sans affecter la sortie",
            "dans le même contexte", "avec une portée locale", "sans toucher les opérateurs",
            "sans altérer les littéraux", "comme note auxiliaire", "pour séparer les idées",
            "sans changer les contrats", "en gardant la clarté", "pour une lecture rapide",
            "sans modifier les décisions", "avec un but documentaire",
        ],
        "project_words_a": ["Aurore", "Brume", "Rivage", "Cèdre", "Lumière", "Prairie", "Quartz", "Sommet", "Nacre", "Sillage"],
        "project_words_b": ["Claire", "Calme", "Dorée", "Lunaire", "Stable", "Verte", "Haute", "Pure", "Douce", "Libre"],
        "names": [
            "Élise", "Lucas", "Camille", "Hugo", "Chloé", "Louis", "Manon", "Jules",
            "Léna", "Arthur", "Inès", "Gabriel", "Clara", "Noé", "Sarah", "Raphaël",
            "Zoé", "Antoine", "Emma", "Maxime", "Alice", "Nathan", "Julie", "Théo",
            "Louise", "Paul", "Anaïs", "Victor", "Nina", "Martin",
        ],
        "header_project": "Projet",
        "header_team": "Équipe",
    },
    "de": {
        "subjects": [
            "das Lesen", "der Ablauf", "die Eingabe", "die Ausgabe", "die Berechnung", "die Bedingung",
            "die Wiederholung", "die Funktion", "der Block", "die Struktur", "die Prüfung",
            "das Feld", "das Objekt", "die Zeichenkette", "der Index", "der Zähler", "das Ergebnis",
            "die Umwandlung", "der Vergleich", "der Zweig",
        ],
        "actions": [
            "ordnet", "bewahrt", "bereitet vor", "trennt", "sortiert", "beschreibt", "stützt",
            "grenzt ab", "gruppiert", "hebt hervor", "klärt", "führt", "markiert", "trägt",
            "schützt", "erklärt", "platziert", "fasst zusammen", "verstärkt", "zeigt",
        ],
        "objects": [
            "die empfangenen Daten", "den Verarbeitungspfad", "die berechnete Information",
            "das sichtbare Ergebnis", "die lokale Entscheidung", "die Arbeitsfolge",
            "die Programmform", "den Zwischenwert", "den Datenlauf",
            "die erwartete Ausgabe", "das menschliche Lesen", "die Code-Kontinuität",
            "die Wertanalyse", "die Variablenvorbereitung", "den Blockabschluss",
            "die Hauptoperation", "den nächsten Schritt", "den nahen Kontext",
            "die angewandte Transformation", "die Auswertungsreihenfolge",
        ],
        "qualifiers": [
            "ohne die Ausführung zu ändern", "ohne Ergebnisse zu ändern", "als visuelle Hilfe",
            "zur leichteren Prüfung", "mit beschreibender Absicht", "ohne Werte zu verändern",
            "in einem sicheren Bereich", "zur besseren Lesbarkeit", "ohne die Ausgabe zu beeinflussen",
            "im gleichen Kontext", "mit lokalem Umfang", "ohne Operatoren zu berühren",
            "ohne Literale zu verändern", "als Hilfsnotiz", "zur Trennung von Gedanken",
            "ohne Verträge zu ändern", "bei klarer Darstellung", "für schnelles Lesen",
            "ohne Entscheidungen zu ändern", "zu Dokumentationszwecken",
        ],
        "project_words_a": ["Morgenrot", "Nebel", "Hafen", "Zeder", "Licht", "Wiese", "Quarz", "Gipfel", "Silber", "Quelle"],
        "project_words_b": ["Klar", "Ruhig", "Golden", "Mondhell", "Stetig", "Grün", "Hoch", "Rein", "Sanft", "Frei"],
        "names": [
            "Lena", "Felix", "Anna", "Jonas", "Mia", "Leon", "Clara", "Paul",
            "Emma", "Noah", "Sophie", "Finn", "Marie", "Lukas", "Laura", "Ben",
            "Nina", "Elias", "Hanna", "Max", "Lea", "Theo", "Greta", "Jan",
            "Marta", "Oskar", "Ida", "Karl", "Rosa", "Moritz",
        ],
        "header_project": "Projekt",
        "header_team": "Team",
    },
    "pt": {
        "subjects": [
            "a leitura", "o fluxo", "a entrada", "a saída", "o cálculo", "a condição",
            "a iteração", "a função", "o bloco", "a estrutura", "a validação",
            "o vetor", "o objeto", "a cadeia", "o índice", "o contador", "o resultado",
            "a conversão", "a comparação", "o ramo",
        ],
        "actions": [
            "organiza", "mantém", "prepara", "separa", "ordena", "descreve", "apoia",
            "delimita", "agrupa", "destaca", "esclarece", "guia", "marca", "sustenta",
            "protege", "explica", "posiciona", "resume", "reforça", "apresenta",
        ],
        "objects": [
            "os dados recebidos", "a rota de processamento", "a informação calculada",
            "o resultado visível", "a decisão local", "a sequência de trabalho",
            "a forma do programa", "o valor intermediário", "o percurso dos dados",
            "a saída esperada", "a leitura humana", "a continuidade do código",
            "a análise de valores", "a preparação de variáveis", "o fechamento do bloco",
            "a operação principal", "o passo seguinte", "o contexto próximo",
            "a transformação aplicada", "a ordem de avaliação",
        ],
        "qualifiers": [
            "sem modificar a execução", "sem mudar os resultados", "como apoio visual",
            "para facilitar a revisão", "com intenção descritiva", "sem alterar valores",
            "em uma zona segura", "para melhorar a legibilidade", "sem afetar a saída",
            "no mesmo contexto", "com alcance local", "sem tocar operadores",
            "sem alterar literais", "como nota auxiliar", "para separar ideias",
            "sem mudar contratos", "mantendo clareza", "para leitura rápida",
            "sem modificar decisões", "com propósito documental",
        ],
        "project_words_a": ["Aurora", "Bruma", "Porto", "Cedro", "Luz", "Prado", "Quartzo", "Cume", "Prata", "Fonte"],
        "project_words_b": ["Clara", "Calma", "Dourada", "Lunar", "Firme", "Verde", "Alta", "Pura", "Suave", "Livre"],
        "names": [
            "Sofia", "Miguel", "Beatriz", "Duarte", "Inês", "João", "Matilde", "Tomás",
            "Leonor", "Tiago", "Mariana", "Rafael", "Clara", "Afonso", "Carolina", "Gonçalo",
            "Madalena", "Diogo", "Ana", "Pedro", "Marta", "Francisco", "Rita", "André",
            "Helena", "Vasco", "Laura", "Nuno", "Sara", "Bruno",
        ],
        "header_project": "Projeto",
        "header_team": "Equipe",
    },
    "ru": {
        "subjects": [
            "чтение", "поток", "ввод", "вывод", "вычисление", "условие",
            "итерация", "функция", "блок", "структура", "проверка",
            "массив", "объект", "строка", "индекс", "счетчик", "результат",
            "преобразование", "сравнение", "ветвь",
        ],
        "actions": [
            "упорядочивает", "сохраняет", "готовит", "разделяет", "сортирует", "описывает", "поддерживает",
            "ограничивает", "группирует", "выделяет", "проясняет", "направляет", "помечает", "удерживает",
            "защищает", "объясняет", "размещает", "резюмирует", "усиливает", "представляет",
        ],
        "objects": [
            "полученные данные", "путь обработки", "вычисленную информацию",
            "видимый результат", "локальное решение", "рабочую последовательность",
            "форму программы", "промежуточное значение", "ход данных",
            "ожидаемый вывод", "чтение человеком", "непрерывность кода",
            "анализ значений", "подготовку переменных", "закрытие блока",
            "основную операцию", "следующий шаг", "близкий контекст",
            "примененное преобразование", "порядок оценки",
        ],
        "qualifiers": [
            "без изменения выполнения", "без изменения результатов", "как визуальная опора",
            "для облегчения проверки", "с описательной целью", "без изменения значений",
            "в безопасной области", "для улучшения читаемости", "без влияния на вывод",
            "в том же контексте", "с локальной областью", "без изменения операторов",
            "без изменения литералов", "как вспомогательная заметка", "для разделения идей",
            "без изменения контрактов", "с сохранением ясности", "для быстрого чтения",
            "без изменения решений", "с документальной целью",
        ],
        "project_words_a": ["Аврора", "Туман", "Гавань", "Кедр", "Свет", "Луг", "Кварц", "Вершина", "Серебро", "Источник"],
        "project_words_b": ["Ясный", "Тихий", "Золотой", "Лунный", "Твердый", "Зеленый", "Высокий", "Чистый", "Мягкий", "Свободный"],
        "names": [
            "Анна", "Иван", "Мария", "Дмитрий", "Елена", "Алексей", "Ольга", "Никита",
            "София", "Михаил", "Дарья", "Артём", "Полина", "Кирилл", "Вера", "Павел",
            "Ирина", "Роман", "Наталья", "Сергей", "Ксения", "Виктор", "Лидия", "Борис",
            "Татьяна", "Глеб", "Юлия", "Максим", "Алина", "Лев",
        ],
        "header_project": "Проект",
        "header_team": "Команда",
    },
    "ko": {
        "subjects": [
            "읽기", "흐름", "입력", "출력", "계산", "조건",
            "반복", "함수", "블록", "구조", "검증",
            "배열", "객체", "문자열", "인덱스", "카운터", "결과",
            "변환", "비교", "분기",
        ],
        "actions": [
            "정리한다", "유지한다", "준비한다", "분리한다", "배열한다", "설명한다", "돕는다",
            "구분한다", "묶는다", "강조한다", "분명히 한다", "안내한다", "표시한다", "받친다",
            "보호한다", "풀이한다", "배치한다", "요약한다", "강화한다", "제시한다",
        ],
        "objects": [
            "받은 데이터", "처리 경로", "계산된 정보",
            "보이는 결과", "로컬 결정", "작업 순서",
            "프로그램 형태", "중간 값", "데이터 이동",
            "예상 출력", "사람의 읽기", "코드 연속성",
            "값 분석", "변수 준비", "블록 마감",
            "주요 연산", "다음 단계", "가까운 문맥",
            "적용된 변환", "평가 순서",
        ],
        "qualifiers": [
            "실행을 바꾸지 않고", "결과를 바꾸지 않고", "시각적 도움으로",
            "검토를 쉽게 하기 위해", "설명 의도로", "값을 바꾸지 않고",
            "안전한 영역에서", "가독성을 높이기 위해", "출력에 영향 없이",
            "같은 문맥에서", "로컬 범위로", "연산자를 건드리지 않고",
            "리터럴을 바꾸지 않고", "보조 메모로", "생각을 나누기 위해",
            "계약을 바꾸지 않고", "명확성을 유지하며", "빠른 읽기를 위해",
            "결정을 바꾸지 않고", "문서화 목적",
        ],
        "project_words_a": ["새벽", "안개", "항구", "삼나무", "빛", "들판", "수정", "봉우리", "은빛", "샘"],
        "project_words_b": ["맑은", "고요한", "금빛", "달빛", "든든한", "푸른", "높은", "깨끗한", "부드러운", "자유로운"],
        "names": [
            "지민", "서준", "하윤", "도윤", "서연", "민준", "지우", "예준",
            "수아", "시우", "유나", "준서", "채원", "현우", "다은", "건우",
            "소율", "우진", "나은", "태민", "가은", "재현", "은서", "민재",
            "하린", "윤호", "수빈", "동현", "예린", "준호",
        ],
        "header_project": "프로젝트",
        "header_team": "팀",
    },
    "ja": {
        "subjects": [
            "読み取り", "流れ", "入力", "出力", "計算", "条件",
            "反復", "関数", "ブロック", "構造", "検証",
            "配列", "オブジェクト", "文字列", "添字", "カウンタ", "結果",
            "変換", "比較", "分岐",
        ],
        "actions": [
            "整理する", "保つ", "準備する", "分ける", "並べる", "説明する", "支える",
            "区切る", "まとめる", "強調する", "明らかにする", "導く", "示す", "確認する",
            "守る", "解説する", "配置する", "要約する", "補強する", "提示する",
        ],
        "objects": [
            "受け取ったデータ", "処理経路", "計算された情報",
            "見える結果", "局所的な判断", "作業順序",
            "プログラムの形", "中間値", "データの移動",
            "期待される出力", "人間の読み取り", "コードの連続性",
            "値の分析", "変数の準備", "ブロックの終端",
            "主要な操作", "次の段階", "近い文脈",
            "適用された変換", "評価順序",
        ],
        "qualifiers": [
            "実行を変えずに", "結果を変えずに", "視覚的な補助として",
            "確認を容易にするため", "説明の意図で", "値を変えずに",
            "安全な領域で", "可読性を高めるため", "出力に影響せずに",
            "同じ文脈で", "局所的な範囲で", "演算子に触れずに",
            "リテラルを変えずに", "補助的な注記として", "考えを分けるため",
            "契約を変えずに", "明確さを保ちながら", "素早い読解のため",
            "判断を変えずに", "文書化の目的で",
        ],
        "project_words_a": ["暁", "霧", "港", "杉", "光", "野原", "水晶", "峰", "銀", "泉"],
        "project_words_b": ["澄んだ", "静かな", "黄金の", "月の", "確かな", "緑の", "高い", "清い", "柔らかな", "自由な"],
        "names": [
            "陽菜", "蓮", "結衣", "悠真", "葵", "湊", "凛", "大翔",
            "美咲", "颯太", "彩乃", "樹", "芽依", "陸", "琴音", "翔",
            "莉子", "直樹", "花音", "悠人", "紗季", "拓海", "七海", "健太",
            "愛", "優斗", "真央", "隼人", "千尋", "和也",
        ],
        "header_project": "プロジェクト",
        "header_team": "チーム",
    },
    "hi": {
        "subjects": [
            "पठन", "प्रवाह", "इनपुट", "आउटपुट", "गणना", "शर्त",
            "दोहराव", "फ़ंक्शन", "खंड", "संरचना", "जांच",
            "सरणी", "वस्तु", "स्ट्रिंग", "सूचकांक", "गणक", "परिणाम",
            "रूपांतरण", "तुलना", "शाखा",
        ],
        "actions": [
            "व्यवस्थित करता है", "बनाए रखता है", "तैयार करता है", "अलग करता है", "क्रम देता है", "वर्णन करता है", "सहारा देता है",
            "सीमा देता है", "समूहित करता है", "उभारता है", "स्पष्ट करता है", "मार्ग दिखाता है", "चिह्नित करता है", "थामता है",
            "सुरक्षित रखता है", "समझाता है", "रखता है", "सार देता है", "मजबूत करता है", "प्रस्तुत करता है",
        ],
        "objects": [
            "प्राप्त डेटा", "प्रसंस्करण मार्ग", "गणित जानकारी",
            "दृश्य परिणाम", "स्थानीय निर्णय", "कार्य क्रम",
            "प्रोग्राम का रूप", "मध्य मान", "डेटा संचलन",
            "अपेक्षित आउटपुट", "मानव पठन", "कोड निरंतरता",
            "मान विश्लेषण", "चर तैयारी", "खंड समाप्ति",
            "मुख्य क्रिया", "अगला चरण", "निकट संदर्भ",
            "लागू रूपांतरण", "मूल्यांकन क्रम",
        ],
        "qualifiers": [
            "निष्पादन बदले बिना", "परिणाम बदले बिना", "दृश्य सहायता के रूप में",
            "समीक्षा आसान करने के लिए", "वर्णनात्मक उद्देश्य से", "मान बदले बिना",
            "सुरक्षित क्षेत्र में", "पठनीयता बढ़ाने के लिए", "आउटपुट प्रभावित किए बिना",
            "उसी संदर्भ में", "स्थानीय दायरे में", "ऑपरेटर छुए बिना",
            "लिटरल बदले बिना", "सहायक टिप्पणी के रूप में", "विचार अलग करने के लिए",
            "अनुबंध बदले बिना", "स्पष्टता रखते हुए", "त्वरित पठन के लिए",
            "निर्णय बदले बिना", "दस्तावेजी उद्देश्य से",
        ],
        "project_words_a": ["अरुण", "कुहासा", "बंदरगाह", "देवदार", "प्रकाश", "मैदान", "स्फटिक", "शिखर", "रजत", "स्रोत"],
        "project_words_b": ["स्पष्ट", "शांत", "स्वर्ण", "चंद्र", "स्थिर", "हरित", "ऊँचा", "शुद्ध", "कोमल", "मुक्त"],
        "names": [
            "आरव", "सिया", "विवान", "अनन्या", "अर्जुन", "मीरा", "कबीर", "ईशा",
            "रोहन", "काव्या", "आदित्य", "रिया", "निखिल", "तारा", "वेद", "नंदिनी",
            "ध्रुव", "आशा", "करण", "लता", "ईशान", "माया", "राहुल", "प्रिया",
            "अमन", "दीया", "समीर", "नेहा", "मोहन", "गीता",
        ],
        "header_project": "परियोजना",
        "header_team": "दल",
    },
    "zh": {
        "subjects": [
            "读取", "流程", "输入", "输出", "计算", "条件",
            "迭代", "函数", "代码块", "结构", "校验",
            "数组", "对象", "字符串", "索引", "计数器", "结果",
            "转换", "比较", "分支",
        ],
        "actions": [
            "整理", "保持", "准备", "分离", "排序", "描述", "支撑",
            "界定", "归组", "强调", "说明", "引导", "标记", "承接",
            "保护", "解释", "放置", "概括", "加强", "呈现",
        ],
        "objects": [
            "接收的数据", "处理路径", "计算出的信息",
            "可见结果", "局部决定", "工作顺序",
            "程序形态", "中间值", "数据流动",
            "预期输出", "人工阅读", "代码连续性",
            "数值分析", "变量准备", "代码块结束",
            "主要操作", "下一步骤", "附近上下文",
            "应用的转换", "求值顺序",
        ],
        "qualifiers": [
            "不改变执行", "不改变结果", "作为视觉辅助",
            "为了便于检查", "带有说明意图", "不改变数值",
            "在安全区域内", "为了提高可读性", "不影响输出",
            "在同一上下文中", "以局部范围", "不触及运算符",
            "不改变字面量", "作为辅助说明", "为了分隔想法",
            "不改变契约", "保持清晰", "为了快速阅读",
            "不改变判断", "用于文档目的",
        ],
        "project_words_a": ["晨光", "薄雾", "港湾", "雪松", "光源", "草地", "水晶", "山峰", "银川", "泉眼"],
        "project_words_b": ["清澈", "安静", "金色", "月色", "稳定", "青绿", "高远", "纯净", "柔和", "自由"],
        "names": [
            "明", "丽", "伟", "芳", "强", "娜", "磊", "敏",
            "洋", "静", "勇", "霞", "杰", "婷", "涛", "慧",
            "超", "琳", "健", "雪", "晨", "欣", "浩", "雯",
            "博", "倩", "宁", "晶", "宇", "菲",
        ],
        "header_project": "项目",
        "header_team": "团队",
    },
    "la": {
        "subjects": [
            "lectio", "fluxus", "introitus", "exitus", "computatio", "condicio",
            "iteratio", "functio", "blocum", "structura", "probatio",
            "series", "obiectum", "catena", "index", "numerator", "eventus",
            "conversio", "comparatio", "ramus",
        ],
        "actions": [
            "ordinat", "servat", "praeparat", "separat", "disponit", "describit", "sustinet",
            "terminat", "congregat", "illustrat", "clarificat", "ducit", "notat", "portat",
            "custodit", "explicat", "ponit", "summat", "confirmat", "ostendit",
        ],
        "objects": [
            "data accepta", "iter processus", "notitiam computatam",
            "eventum visibilem", "iudicium locale", "ordinem operis",
            "formam programmatis", "valorem medium", "cursum datorum",
            "exitum exspectatum", "lectionem humanam", "continuitatem codicis",
            "analysin valorum", "praeparationem variabilium", "finem bloci",
            "operationem principalem", "gradum proximum", "contextum propinquum",
            "conversionem adhibitam", "ordinem aestimationis",
        ],
        "qualifiers": [
            "sine mutatione exsecutionis", "sine mutatione eventuum", "ut auxilium visibile",
            "ad recensionem faciliorem", "cum consilio descriptivo", "sine mutatione valorum",
            "in regione tuta", "ad legendum facilius", "sine effectu in exitu",
            "in eodem contextu", "ambitu locali", "sine tactu operatorum",
            "sine mutatione literalium", "ut nota auxiliaris", "ad ideas separandas",
            "sine mutatione pactorum", "claritate servata", "ad lectionem celerem",
            "sine mutatione iudiciorum", "causa documentaria",
        ],
        "project_words_a": ["Aurora", "Nebula", "Portus", "Cedrus", "Lumen", "Pratum", "Crystallum", "Vertex", "Argentum", "Fons"],
        "project_words_b": ["Clarus", "Quietus", "Aureus", "Lunaris", "Firmus", "Viridis", "Altus", "Purus", "Mollis", "Liber"],
        "names": [
            "Marcus", "Julia", "Lucius", "Claudia", "Gaius", "Aemilia", "Titus", "Livia",
            "Quintus", "Cornelia", "Sextus", "Flavia", "Publius", "Aurelia", "Decimus", "Octavia",
            "Servius", "Valeria", "Manius", "Tullia", "Numerius", "Marcia", "Appius", "Antonia",
            "Tiberius", "Fabia", "Cnaeus", "Horatia", "Aulus", "Domitia",
        ],
        "header_project": "Proiectum",
        "header_team": "Turma",
    },
}

# ============================================================
# Expansión semántica a 100 variantes por elemento
# ============================================================

HEADER_START_PROB = 0.45          # Probabilidad de poner bloque al inicio
FOOTER_IF_NO_HEADER_PROB = 0.2   # Si no hubo header, baja probabilidad de bloque final
INLINE_SPECIAL_PROB = 0.07       # Pokemon/caras/emojis/tokens: bajo
SECTION_SEPARATOR_PROB = 0.1    # Separadores de sección: bajo
TRAILING_COMMENT_PROB = 0.01      # Comentarios inline: bajo para no romper sintaxis
COMMENT_BEFORE_LINE_PROB = 0.08
EXTRA_LINES_AFTER_PROB = 0.07

HEADER_VARIANT_WEIGHTS = {
    "project_only": 0.18,
    "project_team": 0.30,
    "team_only": 0.14,
    "project_team_ids": 0.24,
    "team_ids": 0.14,
}

HEADER_LABELS = {
    "es": {"project": "Proyecto", "team": "Integrantes", "ids": "Matrículas"},
    "en": {"project": "Project", "team": "Team members", "ids": "Institute IDs"},
    "fr": {"project": "Projet", "team": "Membres", "ids": "Matricules"},
    "de": {"project": "Projekt", "team": "Mitglieder", "ids": "Matrikelnummern"},
    "pt": {"project": "Projeto", "team": "Integrantes", "ids": "Matrículas"},
    "ru": {"project": "Проект", "team": "Участники", "ids": "Номера"},
    "ko": {"project": "프로젝트", "team": "구성원", "ids": "학번"},
    "ja": {"project": "プロジェクト", "team": "メンバー", "ids": "登録番号"},
    "hi": {"project": "परियोजना", "team": "सदस्य", "ids": "पंजीकरण"},
    "zh": {"project": "项目", "team": "成员", "ids": "学号"},
    "la": {"project": "Proiectum", "team": "Sodales", "ids": "Numeri"},
}

LAST_NAMES = {
    "es": ["García", "López", "Martínez", "Sánchez", "Ramírez", "Torres", "Flores", "Rivera", "Morales", "Castillo",
           "Vargas", "Herrera", "Rojas", "Mendoza", "Cortés", "Navarro", "Paredes", "Salinas", "Aguilar", "Campos"],
    "en": ["Smith", "Johnson", "Miller", "Brown", "Davis", "Wilson", "Taylor", "Moore", "Clark", "Walker",
           "Hall", "Young", "Allen", "King", "Wright", "Scott", "Green", "Adams", "Baker", "Carter"],
    "fr": ["Martin", "Bernard", "Dubois", "Thomas", "Robert", "Richard", "Petit", "Durand", "Leroy", "Moreau",
           "Simon", "Laurent", "Lefèvre", "Michel", "Garcia", "David", "Bertrand", "Roux", "Vincent", "Fournier"],
    "de": ["Müller", "Schmidt", "Schneider", "Fischer", "Weber", "Meyer", "Wagner", "Becker", "Hoffmann", "Schulz",
           "Koch", "Bauer", "Richter", "Klein", "Wolf", "Schröder", "Neumann", "Schwarz", "Zimmermann", "Braun"],
    "pt": ["Silva", "Santos", "Oliveira", "Souza", "Pereira", "Costa", "Rodrigues", "Almeida", "Nascimento", "Lima",
           "Araújo", "Fernandes", "Carvalho", "Gomes", "Martins", "Rocha", "Ribeiro", "Barbosa", "Mendes", "Cardoso"],
    "ru": ["Иванов", "Петров", "Смирнов", "Кузнецов", "Попов", "Васильев", "Соколов", "Михайлов", "Новиков", "Фёдоров",
           "Морозов", "Волков", "Алексеев", "Лебедев", "Семёнов", "Егоров", "Павлов", "Козлов", "Степанов", "Орлов"],
    "ko": ["김", "이", "박", "최", "정", "강", "조", "윤", "장", "임",
           "한", "오", "서", "신", "권", "황", "안", "송", "류", "홍"],
    "ja": ["佐藤", "鈴木", "高橋", "田中", "伊藤", "渡辺", "山本", "中村", "小林", "加藤",
           "吉田", "山田", "佐々木", "山口", "松本", "井上", "木村", "林", "清水", "山崎"],
    "hi": ["शर्मा", "वर्मा", "गुप्ता", "सिंह", "कुमार", "दास", "मेहता", "जोशी", "चौधरी", "मिश्रा",
           "पटेल", "यादव", "नायर", "राव", "रेड्डी", "कपूर", "भट्ट", "अय्यर", "शेख", "खान"],
    "zh": ["王", "李", "张", "刘", "陈", "杨", "赵", "黄", "周", "吴",
           "徐", "孙", "胡", "朱", "高", "林", "何", "郭", "马", "罗"],
    "la": ["Aelius", "Cornelius", "Julius", "Claudius", "Valerius", "Fabius", "Aurelius", "Tullius", "Flavius", "Marcius",
           "Horatius", "Antonius", "Domitius", "Sergius", "Livius", "Cassius", "Vergilius", "Octavius", "Papirius", "Licinius"],
}

FISH_WORDS = {
    "es": ["Dorado", "Atún", "Mero", "Salmón", "PezLuna", "Trucha", "Róbalo", "Sardina", "Carpa", "Marlin"],
    "en": ["Tuna", "Salmon", "Marlin", "Trout", "Cod", "Carp", "Bass", "Herring", "Mackerel", "Perch"],
    "fr": ["Saumon", "Thon", "Truite", "Carpe", "Mérou", "Sardine", "Bar", "Dorade", "Merlan", "Perche"],
    "de": ["Lachs", "Thun", "Forelle", "Karpfen", "Barsch", "Hering", "Makrele", "Dorsch", "Zander", "Hecht"],
    "pt": ["Salmão", "Atum", "Truta", "Carpa", "Robalo", "Sardinha", "Dourado", "Mero", "Perca", "Bacalhau"],
    "ru": ["Лосось", "Тунец", "Форель", "Карп", "Окунь", "Сельдь", "Щука", "Судак", "Треска", "Сом"],
    "ko": ["연어", "참치", "송어", "잉어", "농어", "청어", "고등어", "대구", "도미", "빙어"],
    "ja": ["鮭", "鮪", "鱒", "鯉", "鱸", "鰯", "鯖", "鱈", "鯛", "鰤"],
    "hi": ["रोहू", "कतला", "हिल्सा", "सुरमई", "माहसीर", "सोल", "ट्राउट", "कार्प", "टूना", "सामन"],
    "zh": ["鲑鱼", "金枪鱼", "鳟鱼", "鲤鱼", "鲈鱼", "沙丁鱼", "鳕鱼", "鲭鱼", "鲷鱼", "鲱鱼"],
    "la": ["Salmo", "Thunnus", "Trutta", "Cyprinus", "Percus", "Clupea", "Scomber", "Gadus", "Sparus", "Muraena"],
}

TECH_CONCEPTS = ["Vector", "Kernel", "Matrix", "Node", "Signal", "Packet", "Circuit", "Lambda", "Cache", "Quantum",
                 "Pixel", "Cipher", "Module", "Runtime", "Parser", "Thread", "Stack", "Bridge", "Orbit", "Flux"]

LOCATIONS = {
    "es": ["Valle", "Bosque", "Río", "Montaña", "Isla", "Puerto", "Caverna", "Torre", "Llanura", "Jardín"],
    "en": ["Valley", "Forest", "River", "Mountain", "Island", "Harbor", "Cavern", "Tower", "Plain", "Garden"],
    "fr": ["Vallée", "Forêt", "Rivière", "Montagne", "Île", "Port", "Caverne", "Tour", "Plaine", "Jardin"],
    "de": ["Tal", "Wald", "Fluss", "Berg", "Insel", "Hafen", "Höhle", "Turm", "Ebene", "Garten"],
    "pt": ["Vale", "Floresta", "Rio", "Montanha", "Ilha", "Porto", "Caverna", "Torre", "Planície", "Jardim"],
    "ru": ["Долина", "Лес", "Река", "Гора", "Остров", "Гавань", "Пещера", "Башня", "Равнина", "Сад"],
    "ko": ["계곡", "숲", "강", "산", "섬", "항구", "동굴", "탑", "평원", "정원"],
    "ja": ["谷", "森", "川", "山", "島", "港", "洞窟", "塔", "平原", "庭"],
    "hi": ["घाटी", "वन", "नदी", "पर्वत", "द्वीप", "बंदरगाह", "गुफा", "मीनार", "मैदान", "उद्यान"],
    "zh": ["山谷", "森林", "河流", "高山", "岛屿", "港湾", "洞穴", "高塔", "平原", "花园"],
    "la": ["Vallis", "Silva", "Flumen", "Mons", "Insula", "Portus", "Specus", "Turris", "Campus", "Hortus"],
}

FANTASY_CONCEPTS = ["Rune", "Dragon", "Crystal", "Moon", "Elder", "Mystic", "Oracle", "Phoenix", "Arcane", "Crown",
                    "Ember", "Aether", "Shadow", "Star", "Dream", "Frost", "Relic", "Warden", "Glyph", "Throne"]

POKEMON_ITEMS = [
    "Pikachu", "Bulbasaur", "Charmander", "Squirtle", "Eevee", "Snorlax", "Mew", "Mewtwo", "Lucario", "Gengar",
    "Jigglypuff", "Psyduck", "Meowth", "Dragonite", "Lapras", "Vaporeon", "Jolteon", "Flareon", "Umbreon", "Espeon",
    "Chikorita", "Cyndaquil", "Totodile", "Treecko", "Torchic", "Mudkip", "Ralts", "Absol", "Bagon", "Metagross",
    "Turtwig", "Chimchar", "Piplup", "Riolu", "Zorua", "Froakie", "Greninja", "Rowlet", "Litten", "Mimikyu",
]

FACE_ITEMS = [
    ":)", ":D", ":P", ":|", ":/", ":3", ":-)", ":-D", ":-P", ";)",
    "^_^", "-_-", "o_o", "O_O", "T_T", ">_<", ":v", ":')", ":]", ":[",
    "(^_^)", "(>_<)", "(o_o)", "(^-^)", "(._.)", "(¬_¬)", "(•_•)", "(^o^)", "(x_x)", "(^.^)",
    "\\o/", "¯\\_(ツ)_/¯", "(╯°□°）╯", "ಠ_ಠ", "ʕ•ᴥ•ʔ", "ᕕ( ᐛ )ᕗ", "(ง'̀-'́)ง", "(づ｡◕‿‿◕｡)づ", "(ಥ﹏ಥ)", "(＾▽＾)",
]

EMOJI_ITEMS = [
    "🙂", "😀", "😄", "😉", "😎", "🤔", "🧠", "✨", "🌟", "🔥",
    "🌙", "☀️", "🌊", "🍃", "🌲", "🐟", "🐉", "🦊", "🐱", "🐶",
    "🚀", "🛰️", "💡", "🔧", "🧩", "📌", "📘", "📝", "🔍", "⚙️",
    "🎯", "🛡️", "🔮", "🪐", "🌈", "🍀", "🦉", "🐢", "🦋", "🏔️",
]

RANDOM_TOKEN_ITEMS = [
    "A7kL9pQ2zX", "mN4vB8cR1t", "Z3xT6qW9sA", "p8Lr2Vn5Qe", "K9dF1sH6jM",
    "q2We7Rt9Yu", "B5nM3xC8vZ", "h7Jk2Lm9Np", "V1bN6mQ4zT", "c9Xy3Za7Wq",
    "R8tY2uI5op", "s4Df6Gh8Jk", "L0pO9iU8yT", "M3nB7vC2xZ", "Q6wE1rT5yU",
    "F2gH8jK4lP", "N9mB3vC7xA", "T5yU1iO6pQ", "D4fG9hJ2kL", "Z8xC3vB7nM",
    "P1oI6uY2tR", "W7eR3tY9uI", "G5hJ1kL8pO", "X2cV6bN4mQ", "Y9uI3oP7aS",
    "E4rT8yU2iO", "H6jK1lP5qW", "C7vB2nM9aZ", "I3oP8aS4dF", "U1iO6pQ2wE",
    "J9kL4pO8iU", "V6bN1mQ5zX", "A2sD7fG3hJ", "R4tY8uI2oP", "M5nB9vC3xZ",
    "Q1wE6rT2yU", "L8pO3iU7yT", "K4lP9qW1eR", "S7dF2gH6jK", "N3mB8vC4xZ",
]

assert len(POKEMON_ITEMS) == 40
assert len(FACE_ITEMS) == 40
assert len(EMOJI_ITEMS) == 40
assert len(RANDOM_TOKEN_ITEMS) == 40

USED_PROJECT_NAMES = set()
USED_ORIGINAL_PROJECT_NAMES = set()

# ============================================================
# Expansión simple a 100 variantes por elemento
# ============================================================

SHORT_PREDICATES = {
    "es": [
        "al inicio", "en el bloque", "antes de salir", "en la lectura", "durante el cálculo",
        "en la rama", "por cada valor", "en el ciclo", "con la entrada", "para la salida",
    ],
    "en": [
        "at the start", "in the block", "before output", "during reading", "during calculation",
        "in the branch", "for each value", "inside the loop", "with the input", "for the output",
    ],
    "fr": [
        "au début", "dans le bloc", "avant la sortie", "pendant la lecture", "pendant le calcul",
        "dans la branche", "pour chaque valeur", "dans la boucle", "avec l'entrée", "pour la sortie",
    ],
    "de": [
        "am Anfang", "im Block", "vor der Ausgabe", "beim Lesen", "bei der Berechnung",
        "im Zweig", "für jeden Wert", "in der Schleife", "mit der Eingabe", "für die Ausgabe",
    ],
    "pt": [
        "no início", "no bloco", "antes da saída", "na leitura", "durante o cálculo",
        "no ramo", "para cada valor", "no ciclo", "com a entrada", "para a saída",
    ],
    "ru": [
        "в начале", "в блоке", "перед выводом", "при чтении", "при вычислении",
        "в ветви", "для каждого значения", "в цикле", "с вводом", "для вывода",
    ],
    "ko": [
        "시작에서", "블록 안에서", "출력 전에", "읽는 동안", "계산 중에",
        "분기 안에서", "각 값마다", "반복문 안에서", "입력과 함께", "출력을 위해",
    ],
    "ja": [
        "開始時に", "ブロック内で", "出力前に", "読み取り中に", "計算中に",
        "分岐内で", "各値ごとに", "ループ内で", "入力と共に", "出力のために",
    ],
    "hi": [
        "आरंभ में", "खंड में", "आउटपुट से पहले", "पठन के दौरान", "गणना के दौरान",
        "शाखा में", "हर मान के लिए", "लूप में", "इनपुट के साथ", "आउटपुट के लिए",
    ],
    "zh": [
        "在开始处", "在代码块内", "在输出前", "读取时", "计算时",
        "在分支内", "对每个值", "在循环内", "配合输入", "用于输出",
    ],
    "la": [
        "in initio", "in bloco", "ante exitum", "in lectione", "in computatione",
        "in ramo", "pro quolibet valore", "in circulo", "cum introitu", "pro exitu",
    ],
}


def expand_to_100(items, lang, kind):
    """
    Crea 100 variantes únicas por elemento, manteniendo frases cortas.
    Evita estructuras raras como 'bien organiza' o predicados con números.
    """
    unique = list(dict.fromkeys(items))

    if len(unique) < 20:
        raise ValueError(
            f"{lang}.{kind} needs 20 unique base items, got {len(unique)}. "
            f"Items: {unique}"
        )

    subject_marks = {
        "es": ["", "actual", "principal", "local", "interno"],
        "en": ["", "current", "main", "local", "inner"],
        "fr": ["", "actuel", "principal", "local", "interne"],
        "de": ["", "aktuell", "zentral", "lokal", "intern"],
        "pt": ["", "atual", "principal", "local", "interno"],
        "ru": ["", "текущий", "основной", "локальный", "внутренний"],
        "ko": ["", "현재", "주요", "로컬", "내부"],
        "ja": ["", "現在の", "主要な", "局所的な", "内部の"],
        "hi": ["", "वर्तमान", "मुख्य", "स्थानीय", "आंतरिक"],
        "zh": ["", "当前", "主要", "局部", "内部"],
        "la": ["", "praesens", "principalis", "localis", "internus"],
    }

    action_marks = {
        "es": ["", "bien", "aquí", "ahora", "también"],
        "en": ["", "well", "here", "now", "also"],
        "fr": ["", "bien", "ici", "maintenant", "aussi"],
        "de": ["", "gut", "hier", "jetzt", "auch"],
        "pt": ["", "bem", "aqui", "agora", "também"],
        "ru": ["", "хорошо", "здесь", "сейчас", "также"],
        "ko": ["", "잘", "여기서", "지금", "또한"],
        "ja": ["", "よく", "ここで", "今", "また"],
        "hi": ["", "अच्छी तरह", "यहाँ", "अब", "भी"],
        "zh": ["", "良好地", "这里", "现在", "也"],
        "la": ["", "bene", "hic", "nunc", "quoque"],
    }

    object_marks = {
        "es": ["", "clave", "local", "actual", "final"],
        "en": ["", "key", "local", "current", "final"],
        "fr": ["", "clé", "local", "actuel", "final"],
        "de": ["", "wichtig", "lokal", "aktuell", "final"],
        "pt": ["", "chave", "local", "atual", "final"],
        "ru": ["", "ключевой", "локальный", "текущий", "итоговый"],
        "ko": ["", "핵심", "로컬", "현재", "최종"],
        "ja": ["", "重要な", "局所的な", "現在の", "最終の"],
        "hi": ["", "मुख्य", "स्थानीय", "वर्तमान", "अंतिम"],
        "zh": ["", "关键", "局部", "当前", "最终"],
        "la": ["", "clavis", "locale", "praesens", "finale"],
    }

    predicate_contexts = {
        "es": ["", "cerca", "breve", "activo", "simple", "estable", "claro", "directo", "interno", "final"],
        "en": ["", "nearby", "brief", "active", "simple", "steady", "clear", "direct", "inner", "final"],
        "fr": ["", "proche", "bref", "actif", "simple", "stable", "clair", "direct", "interne", "final"],
        "de": ["", "nah", "kurz", "aktiv", "einfach", "stabil", "klar", "direkt", "intern", "final"],
        "pt": ["", "próximo", "breve", "ativo", "simples", "estável", "claro", "direto", "interno", "final"],
        "ru": ["", "рядом", "кратко", "активно", "просто", "устойчиво", "ясно", "прямо", "внутри", "итогово"],
        "ko": ["", "가까이", "짧게", "활성", "단순", "안정", "명확", "직접", "내부", "최종"],
        "ja": ["", "近く", "短く", "有効", "単純", "安定", "明確", "直接", "内部", "最終"],
        "hi": ["", "निकट", "संक्षिप्त", "सक्रिय", "सरल", "स्थिर", "स्पष्ट", "सीधा", "आंतरिक", "अंतिम"],
        "zh": ["", "邻近", "简短", "有效", "简单", "稳定", "清晰", "直接", "内部", "最终"],
        "la": ["", "prope", "breviter", "active", "simplex", "stabile", "clare", "directe", "interne", "finale"],
    }

    expanded = []
    seen = set()

    for item in unique:
        if kind == "actions":
            marks = action_marks[lang]
        elif kind == "objects":
            marks = object_marks[lang]
        elif kind == "qualifiers":
            marks = predicate_contexts[lang]
        else:
            marks = subject_marks[lang]

        for mark in marks:
            if kind == "actions":
                if not mark:
                    candidate = item
                elif lang in {"zh", "ja", "ko"}:
                    candidate = f"{mark}{item}"
                else:
                    candidate = f"{item} {mark}"

            elif kind == "qualifiers":
                base_pred = SHORT_PREDICATES[lang][len(expanded) % len(SHORT_PREDICATES[lang])]
                if not mark:
                    candidate = base_pred
                elif lang in {"zh", "ja", "ko"}:
                    candidate = f"{base_pred}{mark}"
                else:
                    candidate = f"{base_pred} {mark}"

            elif not mark:
                candidate = item

            elif lang in {"zh", "ja", "ko"}:
                candidate = f"{mark}{item}"

            else:
                candidate = f"{mark} {item}"

            candidate = " ".join(str(candidate).split())

            if candidate not in seen:
                seen.add(candidate)
                expanded.append(candidate)

    if len(expanded) < MIN_VARIANTS_PER_ELEMENT:
        raise ValueError(
            f"{lang} {kind} only has {len(expanded)} variants. "
            f"Unique base items: {len(unique)}."
        )

    return expanded[:MIN_VARIANTS_PER_ELEMENT]

COMMENT_COMPONENTS = {}

for lang, base in LANG_BASE.items():
    COMMENT_COMPONENTS[lang] = {
        "subjects": expand_to_100(base["subjects"], lang, "subjects"),
        "actions": expand_to_100(base["actions"], lang, "actions"),
        "objects": expand_to_100(base["objects"], lang, "objects"),
        "qualifiers": expand_to_100(base["qualifiers"], lang, "qualifiers"),
    }

for lang in TYPE1_LANGUAGES:
    for key in ["subjects", "actions", "objects", "qualifiers"]:
        assert len(COMMENT_COMPONENTS[lang][key]) == 100, (lang, key)

print("Componentes de comentarios OK: 100 variantes por elemento e idioma")

# ============================================================
# Banco de comentarios simples
# ============================================================

def build_comment_bank_for_language(lang, minimum=MIN_PHRASES_PER_LANGUAGE):
    """
    Genera comentarios simples con estas formas:
    1. sujeto
    2. verbo
    3. sujeto + verbo
    4. sujeto + verbo + objeto
    5. sujeto + verbo + objeto + predicado corto
    6. variante propia del idioma, también corta
    """
    comp = COMMENT_COMPONENTS[lang]

    subjects = comp["subjects"]
    actions = comp["actions"]
    objects = comp["objects"]
    predicates = comp["qualifiers"]

    phrases = []
    seen = set()

    def add(text):
        text = " ".join(str(text).split())
        if text not in seen:
            seen.add(text)
            phrases.append(text)

    for i, (s, a, o, p) in enumerate(itertools.product(subjects, actions, objects, predicates)):
        form = i % 6

        if lang == "es":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{a} {o}.",
            ]

        elif lang == "en":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{a} {o}.",
            ]

        elif lang == "fr":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{a} {o}.",
            ]

        elif lang == "de":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{a} {o}.",
            ]

        elif lang == "pt":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{a} {o}.",
            ]

        elif lang == "ru":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{a} {o}.",
            ]

        elif lang == "ko":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{o} {a}.",
            ]

        elif lang == "ja":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{o}を{a}.",
            ]

        elif lang == "hi":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{o} {a}.",
            ]

        elif lang == "zh":
            variants = [
                f"{s}。",
                f"{a}。",
                f"{s}{a}。",
                f"{s}{a}{o}。",
                f"{s}{a}{o}{p}。",
                f"{a}{o}。",
            ]

        elif lang == "la":
            variants = [
                f"{s}.",
                f"{a}.",
                f"{s} {a}.",
                f"{s} {a} {o}.",
                f"{s} {a} {o} {p}.",
                f"{a} {o}.",
            ]

        add(variants[form])

        if len(phrases) >= minimum:
            break

    return phrases[:minimum]


print("Construyendo bancos de comentarios simples...")
COMMENT_BANKS_LARGE = {
    lang: build_comment_bank_for_language(lang, MIN_PHRASES_PER_LANGUAGE)
    for lang in TYPE1_LANGUAGES
}

for lang, bank in COMMENT_BANKS_LARGE.items():
    print(f"{lang} ({LANGUAGE_NAMES[lang]}): {len(bank)} frases únicas")

assert all(len(bank) >= MIN_PHRASES_PER_LANGUAGE for bank in COMMENT_BANKS_LARGE.values())
print("Bancos simples OK")

# ============================================================
# Utilidades de unicidad
# ============================================================

def stable_rng_for_file(dataset_name, file_id, seed=TYPE1_LOCAL_SEED):
    key = f"{seed}|{dataset_name}|{file_id}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    value = int(digest[:16], 16)
    return random.Random(value)


def choose_language_for_file(dataset_name, file_id):
    key = f"language|{TYPE1_LOCAL_SEED}|{dataset_name}|{file_id}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    idx = int(digest[:8], 16) % len(TYPE1_LANGUAGES)
    return TYPE1_LANGUAGES[idx]


def weighted_choice(rng, weights):
    total = sum(weights.values())
    r = rng.random() * total
    upto = 0
    for key, weight in weights.items():
        upto += weight
        if upto >= r:
            return key
    return list(weights.keys())[-1]


def random_acronym(rng):
    consonants = "BCDFGHJKLMNPQRSTVWXYZ"
    vowels = "AEIOU"
    length = rng.randint(3, 6)
    chars = []
    prev_vowel = False

    for _ in range(length):
        if prev_vowel:
            ch = rng.choice(consonants)
        else:
            ch = rng.choice(consonants + vowels)
        prev_vowel = ch in vowels
        chars.append(ch)

    return "".join(chars)


def original_project_word(rng):
    syllables = ["Hex", "Ari", "Nex", "Vor", "Lum", "Zer", "Kai", "Mor", "Eli", "Ryn",
                 "Sola", "Nyx", "Orin", "Vale", "Quin", "Tera", "Zora", "Mira", "Faron", "Lexa"]
    for _ in range(1000):
        name = rng.choice(syllables) + rng.choice(syllables).lower()
        if name not in USED_ORIGINAL_PROJECT_NAMES:
            USED_ORIGINAL_PROJECT_NAMES.add(name)
            return name
    return "Hexia" + hashlib.sha256(str(rng.random()).encode()).hexdigest()[:6]


def unique_project_name(rng, lang):
    category = weighted_choice(rng, {
        "fish_tech": 0.28,
        "location_fantasy": 0.26,
        "original": 0.28,
        "acronym": 0.18,
    })

    for _ in range(1000):
        if category == "fish_tech":
            candidate = f"{rng.choice(FISH_WORDS[lang])} {rng.choice(TECH_CONCEPTS)}"
        elif category == "location_fantasy":
            candidate = f"{rng.choice(LOCATIONS[lang])} {rng.choice(FANTASY_CONCEPTS)}"
        elif category == "original":
            candidate = original_project_word(rng)
        else:
            candidate = random_acronym(rng)

        if rng.random() < 0.72:
            candidate = f"{candidate} {rng.randint(10, 9999)}"

        if candidate not in USED_PROJECT_NAMES:
            USED_PROJECT_NAMES.add(candidate)
            return candidate

    candidate = f"{random_acronym(rng)} {hashlib.sha256(str(rng.random()).encode()).hexdigest()[:8]}"
    USED_PROJECT_NAMES.add(candidate)
    return candidate


def choose_team_members(rng, lang):
    names = LANG_BASE[lang]["names"]
    surnames = LAST_NAMES[lang]
    count = rng.randint(1, 5)

    members = []
    seen = set()

    while len(members) < count:
        if lang in {"zh", "ko", "ja"}:
            full = f"{rng.choice(surnames)}{rng.choice(names)}"
        else:
            full = f"{rng.choice(names)} {rng.choice(surnames)}"

        if full not in seen:
            seen.add(full)
            members.append(full)

    return members


def choose_student_id_style(rng):
    letters_count = rng.randint(1, 3)
    digits_count = rng.randint(6, 9)
    return letters_count, digits_count


def make_student_id(rng, style):
    letters_count, digits_count = style
    letters = "".join(rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(letters_count))
    digits = "".join(rng.choice("0123456789") for _ in range(digits_count))
    return letters + digits


def maybe_add_special_tail(rng, text):
    if rng.random() >= INLINE_SPECIAL_PROB:
        return text

    category = rng.choice(["pokemon", "face", "emoji", "token"])

    if category == "pokemon":
        item = rng.choice(POKEMON_ITEMS)
    elif category == "face":
        item = rng.choice(FACE_ITEMS)
    elif category == "emoji":
        item = rng.choice(EMOJI_ITEMS)
    else:
        item = rng.choice(RANDOM_TOKEN_ITEMS)

    return f"{text} {item}"


def random_comment_text(rng, lang):
    text = rng.choice(COMMENT_BANKS_LARGE[lang])
    return maybe_add_special_tail(rng, text)


def make_line_comment(rng, lang):
    return "// " + random_comment_text(rng, lang)


def make_block_comment(rng, lang):
    if rng.random() < 0.72:
        return f"/* {random_comment_text(rng, lang)} */"

    return (
        "/*\n"
        f" * {random_comment_text(rng, lang)}\n"
        f" * {random_comment_text(rng, lang)}\n"
        " */"
    )


SEPARATOR_STYLES = [
    lambda text: f"// ---------- {text} ----------",
    lambda text: f"// ========== {text} ==========",
    lambda text: f"// ~~~~~~~~~~ {text} ~~~~~~~~~~",
    lambda text: f"// ########## {text} ##########",
    lambda text: f"// .......... {text} ..........",
    lambda text: f"/* ---------- {text} ---------- */",
    lambda text: f"/* ========== {text} ========== */",
    lambda text: f"/* ~~~~~~~~~~ {text} ~~~~~~~~~~ */",
    lambda text: f"/*\n * {text}\n */",
    lambda text: f"// >>> {text} <<<",
]


def make_optional_separator(rng, lang):
    return rng.choice(SEPARATOR_STYLES)(random_comment_text(rng, lang))


HEADER_STYLES = [
    lambda rows: ["/*", *[f" * {r}" for r in rows], " */"],
    lambda rows: ["// " + " | ".join(rows)],
    lambda rows: ["// " + "=" * 64, *[f"// {r}" for r in rows], "// " + "=" * 64],
    lambda rows: ["// " + "-" * 64, *[f"// {r}" for r in rows], "// " + "-" * 64],
    lambda rows: ["/* " + " :: ".join(rows) + " */"],
    lambda rows: ["/*", " * " + "~" * 56, *[f" * {r}" for r in rows], " * " + "~" * 56, " */"],
    lambda rows: ["// >>> " + rows[0], *[f"//     {r}" for r in rows[1:]], "// <<<"],
    lambda rows: ["/*", " * " + "#" * 56, *[f" * # {r}" for r in rows], " * " + "#" * 56, " */"],
    lambda rows: ["// [" + rows[0] + "]", *[f"// [{i:02d}] {r}" for i, r in enumerate(rows[1:], start=1)]],
    lambda rows: ["/*", *[f" * >>> {r}" for r in rows], " */"],
    lambda rows: ["// " + "." * 64, *[f"// {r}" for r in rows], "// " + "." * 64],
    lambda rows: ["/* " + " -- ".join(rows) + " */"],
    lambda rows: ["// /" + "/" * 62, *[f"// {r}" for r in rows], "// /" + "/" * 62],
    lambda rows: ["/*", " * " + "=" * 20, *[f" * {r}" for r in rows], " * " + "=" * 20, " */"],
    lambda rows: ["// {" + rows[0] + "}", *[f"// - {r}" for r in rows[1:]]],
    lambda rows: ["/*", *[f" * [{i}] {r}" for i, r in enumerate(rows, start=1)], " */"],
    lambda rows: ["// " + "*" * 64, *[f"// {r}" for r in rows], "// " + "*" * 64],
    lambda rows: ["/* " + rows[0] + " */", *[f"// {r}" for r in rows[1:]]],
    lambda rows: ["// <" + rows[0] + ">", *[f"//   {r}" for r in rows[1:]], "// </metadata>"],
    lambda rows: ["/*", " * /\\/\\/\\/\\/\\/\\/\\/\\/\\/\\/\\/\\/\\/\\/\\", *[f" * {r}" for r in rows], " */"],
]

# ============================================================
# 20 estilos estéticos adicionales para encabezados
# ============================================================

import unicodedata

HEADER_BOX_WIDTH = 72


def _comment_safe_text(text):
    """
    Evita romper comentarios de bloque si algún texto trae */ accidentalmente.
    """
    return str(text).replace("*/", "* /").replace("\r", " ").replace("\n", " ").strip()


def _char_width(ch):
    """
    Aproximación para alinear texto con caracteres CJK.
    """
    return 2 if unicodedata.east_asian_width(ch) in {"W", "F"} else 1


def _visible_width(text):
    return sum(_char_width(ch) for ch in str(text))


def _fit_visible(text, width):
    text = _comment_safe_text(text)

    if _visible_width(text) <= width:
        return text

    out = ""
    used = 0

    for ch in text:
        w = _char_width(ch)
        if used + w > max(0, width - 1):
            break
        out += ch
        used += w

    return out + "…"


def _pad_right(text, width):
    text = _fit_visible(text, width)
    return text + " " * max(0, width - _visible_width(text))


def _center_visible(text, width):
    text = _fit_visible(text, width)
    remaining = max(0, width - _visible_width(text))
    left = remaining // 2
    right = remaining - left
    return (" " * left) + text + (" " * right)


def _rows_clean(rows):
    return [_comment_safe_text(r) for r in rows if str(r).strip()]


def _line_comment_box(rows, top_l, top_fill, top_r, mid_l, mid_r, bot_l, bot_fill, bot_r):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 4

    lines = [f"// {top_l}{top_fill * (HEADER_BOX_WIDTH - 2)}{top_r}"]

    for r in rows:
        lines.append(f"// {mid_l} {_pad_right(r, inner)} {mid_r}")

    lines.append(f"// {bot_l}{bot_fill * (HEADER_BOX_WIDTH - 2)}{bot_r}")
    return lines


def _block_comment_box(rows, top_l, top_fill, top_r, mid_l, mid_r, bot_l, bot_fill, bot_r):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 4

    lines = ["/*"]
    lines.append(f" * {top_l}{top_fill * (HEADER_BOX_WIDTH - 2)}{top_r}")

    for r in rows:
        lines.append(f" * {mid_l} {_pad_right(r, inner)} {mid_r}")

    lines.append(f" * {bot_l}{bot_fill * (HEADER_BOX_WIDTH - 2)}{bot_r}")
    lines.append(" */")
    return lines


def _ornament_line(left, fill, middle, right, width=HEADER_BOX_WIDTH):
    middle = f" {middle} "
    side_len = max(2, (width - _visible_width(middle) - 2) // 2)
    return f"{left}{fill * side_len}{middle}{fill * side_len}{right}"


# 1. Caja fina Unicode
def header_style_aesthetic_01(rows):
    return _line_comment_box(rows, "┌", "─", "┐", "│", "│", "└", "─", "┘")


# 2. Caja doble Unicode
def header_style_aesthetic_02(rows):
    return _line_comment_box(rows, "╔", "═", "╗", "║", "║", "╚", "═", "╝")


# 3. Caja pesada Unicode
def header_style_aesthetic_03(rows):
    return _line_comment_box(rows, "┏", "━", "┓", "┃", "┃", "┗", "━", "┛")


# 4. Caja redondeada
def header_style_aesthetic_04(rows):
    return _line_comment_box(rows, "╭", "─", "╮", "│", "│", "╰", "─", "╯")


# 5. Caja ASCII limpia
def header_style_aesthetic_05(rows):
    return _line_comment_box(rows, "+", "-", "+", "|", "|", "+", "-", "+")


# 6. Caja ASCII doble
def header_style_aesthetic_06(rows):
    return _line_comment_box(rows, "+", "=", "+", "|", "|", "+", "=", "+")


# 7. Caja con puntos
def header_style_aesthetic_07(rows):
    return _line_comment_box(rows, ".", ".", ".", ":", ":", "'", ".", "'")


# 8. Caja con estrellas
def header_style_aesthetic_08(rows):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 4

    lines = [
        "// " + _ornament_line("┏", "━", "★", "┓"),
    ]

    for r in rows:
        lines.append(f"// ┃ {_center_visible(r, inner)} ┃")

    lines.append("// " + _ornament_line("┗", "━", "★", "┛"))
    return lines


# 9. Caja con rombos
def header_style_aesthetic_09(rows):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 4

    lines = [
        "// " + _ornament_line("╔", "═", "◆", "╗"),
    ]

    for r in rows:
        lines.append(f"// ║ {_center_visible(r, inner)} ║")

    lines.append("// " + _ornament_line("╚", "═", "◆", "╝"))
    return lines


# 10. Caja con separador central
def header_style_aesthetic_10(rows):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 4

    lines = [
        "// ┌" + "─" * (HEADER_BOX_WIDTH - 2) + "┐",
        "// │" + _center_visible("metadata", HEADER_BOX_WIDTH - 2) + "│",
        "// ├" + "─" * (HEADER_BOX_WIDTH - 2) + "┤",
    ]

    for r in rows:
        lines.append(f"// │ {_pad_right(r, inner)} │")

    lines.append("// └" + "─" * (HEADER_BOX_WIDTH - 2) + "┘")
    return lines


# 11. Bloque elegante fino
def header_style_aesthetic_11(rows):
    return _block_comment_box(rows, "┌", "─", "┐", "│", "│", "└", "─", "┘")


# 12. Bloque elegante doble
def header_style_aesthetic_12(rows):
    return _block_comment_box(rows, "╔", "═", "╗", "║", "║", "╚", "═", "╝")


# 13. Bloque pesado
def header_style_aesthetic_13(rows):
    return _block_comment_box(rows, "┏", "━", "┓", "┃", "┃", "┗", "━", "┛")


# 14. Bloque redondeado
def header_style_aesthetic_14(rows):
    return _block_comment_box(rows, "╭", "─", "╮", "│", "│", "╰", "─", "╯")


# 15. Bloque ASCII estable
def header_style_aesthetic_15(rows):
    return _block_comment_box(rows, "+", "-", "+", "|", "|", "+", "-", "+")


# 16. Banner centrado sin caja cerrada
def header_style_aesthetic_16(rows):
    rows = _rows_clean(rows)
    lines = [
        "// " + "─" * HEADER_BOX_WIDTH,
        "// " + _center_visible("project notes", HEADER_BOX_WIDTH),
        "// " + "─" * HEADER_BOX_WIDTH,
    ]

    for r in rows:
        lines.append("// " + _center_visible(r, HEADER_BOX_WIDTH))

    lines.append("// " + "─" * HEADER_BOX_WIDTH)
    return lines


# 17. Banner diagonal ASCII
def header_style_aesthetic_17(rows):
    rows = _rows_clean(rows)
    lines = [
        "// " + "/" * HEADER_BOX_WIDTH,
    ]

    for r in rows:
        lines.append("// // " + _pad_right(r, HEADER_BOX_WIDTH - 6) + " //")

    lines.append("// " + "/" * HEADER_BOX_WIDTH)
    return lines


# 18. Banner con columnas
def header_style_aesthetic_18(rows):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 8

    lines = [
        "// [::" + "=" * (HEADER_BOX_WIDTH - 6) + "::]",
    ]

    for r in rows:
        lines.append("// [  " + _pad_right(r, inner) + "  ]")

    lines.append("// [::" + "=" * (HEADER_BOX_WIDTH - 6) + "::]")
    return lines


# 19. Bloque de tarjeta compacta
def header_style_aesthetic_19(rows):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 6

    lines = [
        "/*",
        " *  " + "╭" + "─" * (HEADER_BOX_WIDTH - 6) + "╮",
    ]

    for r in rows:
        lines.append(" *  │ " + _pad_right(r, inner - 2) + " │")

    lines.append(" *  " + "╰" + "─" * (HEADER_BOX_WIDTH - 6) + "╯")
    lines.append(" */")
    return lines


# 20. Bloque ornamental minimalista
def header_style_aesthetic_20(rows):
    rows = _rows_clean(rows)
    inner = HEADER_BOX_WIDTH - 4

    lines = [
        "/*",
        " * " + _ornament_line("┍", "━", "✦", "┑"),
    ]

    for r in rows:
        lines.append(" * │ " + _center_visible(r, inner) + " │")

    lines.append(" * " + _ornament_line("┕", "━", "✦", "┙"))
    lines.append(" */")
    return lines


AESTHETIC_HEADER_STYLES = [
    header_style_aesthetic_01,
    header_style_aesthetic_02,
    header_style_aesthetic_03,
    header_style_aesthetic_04,
    header_style_aesthetic_05,
    header_style_aesthetic_06,
    header_style_aesthetic_07,
    header_style_aesthetic_08,
    header_style_aesthetic_09,
    header_style_aesthetic_10,
    header_style_aesthetic_11,
    header_style_aesthetic_12,
    header_style_aesthetic_13,
    header_style_aesthetic_14,
    header_style_aesthetic_15,
    header_style_aesthetic_16,
    header_style_aesthetic_17,
    header_style_aesthetic_18,
    header_style_aesthetic_19,
    header_style_aesthetic_20,
]

assert len(AESTHETIC_HEADER_STYLES) == 20

# Mantiene tus estilos anteriores y agrega estos 20 nuevos.
HEADER_STYLES = HEADER_STYLES + AESTHETIC_HEADER_STYLES

print("HEADER_STYLES total:", len(HEADER_STYLES))


def build_metadata_rows(rng, lang):
    labels = HEADER_LABELS[lang]
    variant = weighted_choice(rng, HEADER_VARIANT_WEIGHTS)

    project_name = unique_project_name(rng, lang)
    team = choose_team_members(rng, lang)
    id_style = choose_student_id_style(rng)
    ids = [make_student_id(rng, id_style) for _ in team]

    rows = []

    if variant in {"project_only", "project_team", "project_team_ids"}:
        rows.append(f"{labels['project']}: {project_name}")

    if variant in {"project_team", "team_only", "project_team_ids", "team_ids"}:
        rows.append(f"{labels['team']}: {', '.join(team)}")

    if variant in {"project_team_ids", "team_ids"}:
        rows.append(f"{labels['ids']}: {', '.join(ids)}")

    return rows, variant


def make_metadata_block(rng, lang):
    rows, variant = build_metadata_rows(rng, lang)
    style = rng.choice(HEADER_STYLES)
    return style(rows), variant


def make_optional_header(rng, lang):
    if rng.random() >= HEADER_START_PROB:
        return [], None

    block, variant = make_metadata_block(rng, lang)

    if rng.random() < 0.55:
        block.append("")

    return block, variant


def make_optional_footer(rng, lang):
    if rng.random() >= FOOTER_IF_NO_HEADER_PROB:
        return [], None

    block, variant = make_metadata_block(rng, lang)

    if rng.random() < 0.55:
        return [""] + block, variant

    return block, variant


def is_safe_for_trailing_comment(line):
    stripped = line.rstrip()

    if not stripped:
        return False

    if stripped.endswith("\\"):
        return False

    if stripped.lstrip().startswith("//"):
        return False

    if stripped.lstrip().startswith("*"):
        return False

    return True

# ============================================================
# Manejo seguro de comentarios originales
# ============================================================

# Para eliminar TODOS los comentarios originales, déjalo en 0.0
# Si algún día quieres conservar algunos con probabilidad baja, usa 0.005 o 0.01
PRESERVE_ORIGINAL_COMMENT_PROB = 0.5

REMOVE_ORIGINAL_COMMENTS = True


REGEX_PREFIX_WORDS = {
    "return", "throw", "case", "delete", "void", "typeof",
    "instanceof", "in", "of", "yield", "await", "else",
    "do", "while", "if", "for", "switch", "with",
}

REGEX_PREFIX_CHARS = {
    "(", "[", "{", "=", ":", ",", ";", "!", "?", "&", "|",
    "^", "~", "+", "-", "*", "%", "<", ">",
}


def _can_start_regex(prev_sig):
    """
    Heurística conservadora para distinguir regex literal de división.
    No es parser JS completo, pero evita borrar // o /* dentro de regex en muchos casos.
    """
    if prev_sig is None:
        return True

    if prev_sig in REGEX_PREFIX_CHARS:
        return True

    if isinstance(prev_sig, str) and prev_sig in REGEX_PREFIX_WORDS:
        return True

    return False


def _is_identifier_start(ch):
    return ch == "_" or ch == "$" or ch.isalpha() or ord(ch) > 127


def _is_identifier_part(ch):
    return ch == "_" or ch == "$" or ch.isalnum() or ord(ch) > 127


def _consume_identifier_or_number(code, i):
    """
    Consume un identificador/palabra/número para actualizar prev_sig.
    No modifica texto; solo ayuda al scanner.
    """
    n = len(code)
    start = i

    if i < n and _is_identifier_start(code[i]):
        i += 1
        while i < n and _is_identifier_part(code[i]):
            i += 1
        return code[start:i], i

    if i < n and code[i].isdigit():
        i += 1
        while i < n and (
            code[i].isalnum()
            or code[i] in "._"
        ):
            i += 1
        return code[start:i], i

    return code[i], i + 1


def strip_js_comments_safely(code, rng=None, keep_prob=0.0):
    """
    Elimina comentarios JS de forma segura.

    Maneja:
    - // line comments
    - /* block comments */
    - strings con '...'
    - strings con "..."
    - template literals `...`
    - expresiones dentro de template literals ${ ... }
    - regex literals /.../
    - shebang #!/usr/bin/env node

    No elimina:
    - // dentro de strings
    - /* */ dentro de strings
    - // o /* */ dentro de regex literals
    - texto dentro de template literal plano

    Devuelve:
    - código sin comentarios
    - comentarios conservados
    - comentarios eliminados
    """
    if rng is None:
        rng = random.Random(0)

    out = []
    i = 0
    n = len(code)

    kept = 0
    removed = 0

    state = "normal"
    quote = None
    prev_sig = None

    template_expr_depth = 0
    regex_in_class = False

    # Shebang: debe seguir siendo primera línea exacta.
    if code.startswith("#!"):
        end = code.find("\n")
        if end == -1:
            return code, 0, 0
        out.append(code[:end + 1])
        i = end + 1

    while i < n:
        ch = code[i]
        nxt = code[i + 1] if i + 1 < n else ""

        # ====================================================
        # NORMAL JS CODE
        # ====================================================
        if state == "normal":
            # Comentario de línea
            if ch == "/" and nxt == "/":
                start = i
                i += 2

                while i < n and code[i] not in "\r\n":
                    i += 1

                comment_text = code[start:i]

                if rng.random() < keep_prob:
                    out.append(comment_text)
                    kept += 1
                else:
                    # Espacio seguro para no unir tokens.
                    out.append(" ")
                    removed += 1

                # No consumimos el salto de línea; se procesa en la próxima vuelta.
                continue

            # Comentario de bloque
            if ch == "/" and nxt == "*":
                start = i
                i += 2

                while i + 1 < n and not (code[i] == "*" and code[i + 1] == "/"):
                    i += 1

                if i + 1 < n:
                    i += 2

                comment_text = code[start:i]

                if rng.random() < keep_prob:
                    out.append(comment_text)
                    kept += 1
                else:
                    # Preservar saltos de línea porque pueden importar para ASI.
                    newline_count = comment_text.count("\n")
                    if newline_count:
                        out.append(" " + ("\n" * newline_count) + " ")
                    else:
                        out.append(" ")
                    removed += 1

                continue

            # String normal
            if ch in ("'", '"'):
                quote = ch
                state = "string"
                out.append(ch)
                i += 1
                continue

            # Template literal
            if ch == "`":
                state = "template"
                out.append(ch)
                i += 1
                continue

            # Regex literal
            if ch == "/" and _can_start_regex(prev_sig):
                state = "regex"
                regex_in_class = False
                out.append(ch)
                i += 1
                continue

            # Tracking de template expression ${ ... }
            if template_expr_depth > 0:
                if ch == "{":
                    template_expr_depth += 1
                    out.append(ch)
                    prev_sig = ch
                    i += 1
                    continue

                if ch == "}":
                    template_expr_depth -= 1
                    out.append(ch)
                    prev_sig = ch
                    i += 1

                    if template_expr_depth == 0:
                        state = "template"

                    continue

            # Whitespace normal
            if ch.isspace():
                out.append(ch)
                i += 1
                continue

            # Identificadores / números
            if _is_identifier_start(ch) or ch.isdigit():
                token, new_i = _consume_identifier_or_number(code, i)
                out.append(code[i:new_i])
                prev_sig = token
                i = new_i
                continue

            # Caracter normal
            out.append(ch)
            prev_sig = ch
            i += 1
            continue

        # ====================================================
        # STRING: '...' or "..."
        # ====================================================
        if state == "string":
            out.append(ch)

            if ch == "\\" and i + 1 < n:
                out.append(code[i + 1])
                i += 2
                continue

            if ch == quote:
                state = "normal"
                quote = None
                prev_sig = "literal"

            i += 1
            continue

        # ====================================================
        # TEMPLATE LITERAL: `...`
        # ====================================================
        if state == "template":
            out.append(ch)

            if ch == "\\" and i + 1 < n:
                out.append(code[i + 1])
                i += 2
                continue

            if ch == "`":
                state = "normal"
                prev_sig = "literal"
                i += 1
                continue

            # Entrar a expresión ${ ... }
            if ch == "$" and nxt == "{":
                out.append(nxt)
                state = "normal"
                template_expr_depth = 1
                prev_sig = "{"
                i += 2
                continue

            i += 1
            continue

        # ====================================================
        # REGEX LITERAL: /.../
        # ====================================================
        if state == "regex":
            out.append(ch)

            if ch == "\\" and i + 1 < n:
                out.append(code[i + 1])
                i += 2
                continue

            if ch == "[":
                regex_in_class = True
                i += 1
                continue

            if ch == "]":
                regex_in_class = False
                i += 1
                continue

            if ch == "/" and not regex_in_class:
                state = "normal"
                prev_sig = "literal"
                i += 1

                # Regex flags
                while i < n and code[i].isalpha():
                    out.append(code[i])
                    i += 1

                continue

            i += 1
            continue

    return "".join(out), kept, removed


def filter_original_comments(code, rng, keep_prob=PRESERVE_ORIGINAL_COMMENT_PROB):
    """
    Wrapper compatible con tu código actual.

    Si REMOVE_ORIGINAL_COMMENTS=True:
    - intenta eliminar comentarios originales
    - conserva 0% si keep_prob=0.0
    - conserva algunos si keep_prob > 0

    La seguridad final la siguen dando:
    - type1_tokens_equivalent(original_code, clone_code)
    - syntax_check_text(clone_code)
    """
    if not REMOVE_ORIGINAL_COMMENTS:
        return code, 0, 0

    stripped_code, kept, removed = strip_js_comments_safely(
        code,
        rng=rng,
        keep_prob=keep_prob,
    )

    return stripped_code, kept, removed

# ============================================================
# Whitespace interno seguro entre tokens
# ============================================================

INTRA_LINE_SPACING_PROB = 0.02

JS_MULTI_TOKENS = [
    ">>>=", "===", "!==", "<<=", ">>=", "**=", "=>", "++", "--",
    "==", "!=", "<=", ">=", "&&", "||", "+=", "-=", "*=", "%=",
    "&=", "|=", "^=", "<<", ">>", "**", "??", "?.", "...",
]

JS_SINGLE_TOKENS = set("()[]{};,?:+-*%<>=!&|^~")


def compute_safe_code_lines(code):
    """
    Marca una línea como segura solo si:
    - empieza en estado normal
    - termina en estado normal
    - no toca template literals
    - no toca comentarios de bloque
    - no está dentro de string multilínea
    """
    safe = []

    state = "normal"
    quote = None
    escape = False

    line_started_state = "normal"
    line_touched_unsafe = False

    i = 0
    n = len(code)

    while i < n:
        ch = code[i]
        nxt = code[i + 1] if i + 1 < n else ""

        if ch == "\n":
            safe.append(
                line_started_state == "normal"
                and state == "normal"
                and not line_touched_unsafe
            )

            line_started_state = state
            line_touched_unsafe = state != "normal"

            i += 1
            continue

        if state == "normal":
            if ch in ("'", '"'):
                state = "string"
                quote = ch
                i += 1
                continue

            if ch == "`":
                state = "template"
                line_touched_unsafe = True
                i += 1
                continue

            if ch == "/" and nxt == "*":
                state = "block_comment"
                line_touched_unsafe = True
                i += 2
                continue

            if ch == "/" and nxt == "/":
                while i < n and code[i] != "\n":
                    i += 1
                continue

            i += 1
            continue

        if state == "string":
            line_touched_unsafe = True

            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == quote:
                state = "normal"
                quote = None

            i += 1
            continue

        if state == "template":
            line_touched_unsafe = True

            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == "`":
                state = "normal"

            i += 1
            continue

        if state == "block_comment":
            line_touched_unsafe = True

            if ch == "*" and nxt == "/":
                state = "normal"
                i += 2
                continue

            i += 1
            continue

    safe.append(
        line_started_state == "normal"
        and state == "normal"
        and not line_touched_unsafe
    )

    return safe


def is_word_like_token(tok):
    if not tok:
        return False

    if tok[0].isalpha() or tok[0] in "_$":
        return all(ch.isalnum() or ch in "_$" for ch in tok)

    if tok[0].isdigit():
        return True

    return False


import re

SCI_NOTATION_RE = re.compile(r"\b\d+(?:\.\d+)?[eE][+-]?\d+\b")


def should_skip_intra_line_spacing(line):
    stripped = line.strip()

    if not stripped:
        return True

    if stripped.startswith("//") or stripped.startswith("*"):
        return True

    if SCI_NOTATION_RE.search(line):
        return True

    risky_chars = ['"', "'", "`", "/", "."]

    return any(ch in line for ch in risky_chars)


def tokenize_simple_js_line(line):
    """
    Tokenizador conservador para líneas sin strings/comentarios/slash.
    No es parser JS completo, pero sirve para spacing seguro en líneas simples.
    """
    tokens = []
    i = 0
    n = len(line)

    multi_sorted = sorted(JS_MULTI_TOKENS, key=len, reverse=True)

    while i < n:
        ch = line[i]

        if ch.isspace():
            i += 1
            continue

        matched = None
        for op in multi_sorted:
            if line.startswith(op, i):
                matched = op
                break

        if matched:
            tokens.append(matched)
            i += len(matched)
            continue

        if ch in JS_SINGLE_TOKENS:
            tokens.append(ch)
            i += 1
            continue

        start = i
        while i < n:
            ch2 = line[i]

            if ch2.isspace():
                break

            if ch2 in JS_SINGLE_TOKENS:
                break

            matched_here = False
            for op in multi_sorted:
                if line.startswith(op, i):
                    matched_here = True
                    break

            if matched_here:
                break

            i += 1

        tokens.append(line[start:i])

    return tokens


def random_gap_between_tokens(prev_tok, curr_tok, rng):
    """
    Decide cuántos espacios poner entre dos tokens.
    Nunca junta tokens cuando eso puede cambiar el tokenizado.
    """
    if prev_tok is None:
        return ""

    if prev_tok in {".", "?.", "..."} or curr_tok in {".", "?.", "..."}:
        return ""

    if prev_tok in {"++", "--"} or curr_tok in {"++", "--"}:
        return ""

    if is_word_like_token(prev_tok) and is_word_like_token(curr_tok):
        return " " * rng.choice([1, 1, 2, 3])

    if prev_tok in {
        "var", "let", "const", "function", "return", "throw", "new",
        "typeof", "void", "delete", "yield", "await"
    }:
        return " " * rng.choice([1, 1, 2])

    if curr_tok == "{":
        return " " * rng.choice([0, 1, 1, 2])

    if prev_tok == "{":
        return rng.choice(["", " ", "  "])

    if prev_tok in {"(", "["}:
        return rng.choice(["", "", " "])

    if curr_tok in {")", "]"}:
        return rng.choice(["", "", " "])

    if curr_tok in {",", ";"}:
        return ""

    if prev_tok in {",", ";"}:
        return rng.choice(["", " ", " ", "  "])

    operator_tokens = {
        "=", "+", "-", "*", "%", "<", ">", "<=", ">=", "==", "===",
        "!=", "!==", "&&", "||", "+=", "-=", "*=", "%=", "&", "|",
        "^", "&=", "|=", "^=", "<<", ">>", ">>>", "<<=", ">>=", ">>>=",
        "**", "**=", "??", "=>",
    }

    if prev_tok in operator_tokens or curr_tok in operator_tokens:
        return rng.choice(["", " ", " ", "  "])

    return rng.choice(["", " ", " "])


def randomize_intra_line_spacing(line, rng):
    """
    Agrega/quita espacios internos entre tokens de una línea simple.
    No toca strings, comentarios ni líneas con slash.
    """
    if should_skip_intra_line_spacing(line):
        return line

    tokens = tokenize_simple_js_line(line)

    if len(tokens) <= 1:
        return line

    out = []
    prev = None

    for tok in tokens:
        out.append(random_gap_between_tokens(prev, tok, rng))
        out.append(tok)
        prev = tok

    return "".join(out)


# ============================================================
# Validación de sintaxis y utilidades defensivas
# ============================================================

ESM_IMPORT_RE = re.compile(
    r"^\s*import\s+(?:[\w*{]|['\"])",
    re.MULTILINE,
)

ESM_EXPORT_RE = re.compile(
    r"^\s*export\s+(?:default|const|let|var|function|class|\{|\*)",
    re.MULTILINE,
)


def looks_like_esm(code):
    """
    Detecta módulos ES para intentar validarlos como .mjs.
    """
    return bool(ESM_IMPORT_RE.search(code) or ESM_EXPORT_RE.search(code))


def syntax_check_auto_mode(code, file_id, prefix, report_dir):
    """
    Intenta validar el código en el modo más razonable:
    - .mjs primero si parece ES Module.
    - .cjs primero si parece script/CommonJS.
    - Si falla el primer modo, prueba el otro.

    Devuelve:
    - ok
    - reason
    - extension usada
    """
    modes = [".mjs", ".cjs"] if looks_like_esm(code) else [".cjs", ".mjs"]

    last_reason = ""

    for ext in modes:
        temp_path = report_dir / f"_{prefix}_{file_id}{ext}"
        ok, reason = syntax_check_text(code, temp_path=temp_path)

        if ok:
            return True, reason, ext

        last_reason = reason

    return False, last_reason, modes[-1]


def safe_type1_tokens_equivalent(original_code, clone_code):
    """
    Wrapper para evitar que una excepción del tokenizador rompa la corrida.
    """
    try:
        return bool(type1_tokens_equivalent(original_code, clone_code)), ""
    except Exception as exc:
        return False, repr(exc)


def has_legacy_html_comment_syntax(code):
    """
    JS tiene casos legacy con <!-- y -->.
    Para esos archivos conviene usar fallback identity.
    """
    return "<!--" in code or "-->" in code


def has_shebang(code):
    return code.startswith("#!")


def normalize_final_newline(code):
    return code if code.endswith("\n") else code + "\n"


def make_type1_clone_identity_fallback(code, dataset_name="", file_id=""):
    """
    Fallback 100% conservador:
    - No agrega header.
    - No cambia comentarios.
    - No cambia indentación.
    - Solo asegura newline final.

    Esto sigue siendo Type I, aunque sea una variante mínima.
    """
    lang = choose_language_for_file(dataset_name, file_id)

    return (
        normalize_final_newline(code),
        lang,
        "identity_fallback",
        None,
        0,
        0,
    )


# ============================================================
# Generador Type I
# ============================================================

def make_type1_clone_locally(code, dataset_name="", file_id=""):
    """
    Genera un Type I local:
    - Conserva tokens ejecutables.
    - Solo cambia whitespace y comentarios.
    - Cada archivo usa una sola lengua.
    - No menciona que es un clon.

    Si el archivo tiene sintaxis legacy peligrosa, usa fallback identity.
    """
    rng = stable_rng_for_file(dataset_name, file_id)
    lang = choose_language_for_file(dataset_name, file_id)

    if has_legacy_html_comment_syntax(code):
        return make_type1_clone_identity_fallback(
            code,
            dataset_name=dataset_name,
            file_id=file_id,
        )

    code_for_body, original_comments_kept, original_comments_removed = filter_original_comments(
        code,
        rng,
        keep_prob=PRESERVE_ORIGINAL_COMMENT_PROB,
    )

    lines = code_for_body.splitlines()
    new_lines = []
    safe_lines = compute_safe_code_lines(code_for_body)

    start_idx = 0
    if lines and lines[0].startswith("#!"):
        new_lines.append(lines[0])
        start_idx = 1

    header_lines = []
    header_variant = None
    footer_variant = None

    # No meter header en archivos con shebang.
    if start_idx == 0:
        header_lines, header_variant = make_optional_header(rng, lang)

    new_lines.extend(header_lines)

    header_comment_count = rng.choice([0, 0, 0, 1])
    for _ in range(header_comment_count):
        new_lines.append(
            make_line_comment(rng, lang)
            if rng.random() < 0.60
            else make_block_comment(rng, lang)
        )
        if rng.random() < 0.35:
            new_lines.append("")

    for idx, line in enumerate(lines[start_idx:], start=start_idx):
        prev_line_is_safe = safe_lines[idx - 1] if idx - 1 >= 0 and idx - 1 < len(safe_lines) else True
        line_is_safe = safe_lines[idx] if idx < len(safe_lines) else False
        next_line_is_safe = safe_lines[idx + 1] if idx + 1 < len(safe_lines) else True

        if not line_is_safe:
            new_lines.append(line)
            continue

        stripped_right = line.rstrip("\n")
        stripped = stripped_right.strip()

        if rng.random() < SECTION_SEPARATOR_PROB and (
            "function" in stripped
            or "=>" in stripped
            or stripped.startswith("class ")
            or stripped.startswith("for ")
            or stripped.startswith("if ")
        ):
            new_lines.append(make_optional_separator(rng, lang))

        if rng.random() < COMMENT_BEFORE_LINE_PROB:
            new_lines.append(
                make_line_comment(rng, lang)
                if rng.random() < 0.65
                else make_block_comment(rng, lang)
            )

        if not stripped:
            if not prev_line_is_safe or not next_line_is_safe:
                new_lines.append(line)
                continue

            blank_count = rng.choice([0, 1, 1, 2])
            for _ in range(blank_count):
                if rng.random() < 0.08:
                    new_lines.append(make_line_comment(rng, lang))
                else:
                    new_lines.append("")
            continue

        leading_style = rng.choice([
            "spaces_small",
            "spaces_medium",
            "tabs",
            "mixed",
            "none",
            "preserve_plus",
        ])

        original_indent_len = len(stripped_right) - len(stripped_right.lstrip(" \t"))
        original_indent = stripped_right[:original_indent_len]
        original_without_indent = stripped_right.lstrip(" \t")

        if rng.random() < INTRA_LINE_SPACING_PROB:
            original_without_indent = randomize_intra_line_spacing(original_without_indent, rng)

        if leading_style == "spaces_small":
            new_indent = " " * rng.randint(0, 4)
        elif leading_style == "spaces_medium":
            new_indent = " " * rng.randint(0, 10)
        elif leading_style == "tabs":
            new_indent = "\t" * rng.randint(0, 2)
        elif leading_style == "mixed":
            new_indent = ("\t" * rng.randint(0, 1)) + (" " * rng.randint(0, 6))
        elif leading_style == "preserve_plus":
            new_indent = original_indent + (" " * rng.randint(0, 4))
        else:
            new_indent = ""

        new_line = new_indent + original_without_indent

        if rng.random() < 0.16:
            new_line += " " * rng.randint(1, 5)

        # Se deja desactivado con TRAILING_COMMENT_PROB = 0.0.
        if rng.random() < TRAILING_COMMENT_PROB and is_safe_for_trailing_comment(new_line):
            new_line += "  " + make_line_comment(rng, lang)

        new_lines.append(new_line)

        if next_line_is_safe and rng.random() < EXTRA_LINES_AFTER_PROB:
            extra_count = rng.choice([1, 1, 2])
            for _ in range(extra_count):
                choice = rng.random()
                if choice < 0.50:
                    new_lines.append("")
                elif choice < 0.82:
                    new_lines.append(make_line_comment(rng, lang))
                else:
                    new_lines.append(make_block_comment(rng, lang))

    if header_variant is None and start_idx == 0:
        footer_lines, footer_variant = make_optional_footer(rng, lang)
        new_lines.extend(footer_lines)

    if lines and (safe_lines[-1] if safe_lines else True) and rng.random() < 0.20:
        new_lines.append("")
        new_lines.append(
            make_line_comment(rng, lang)
            if rng.random() < 0.65
            else make_block_comment(rng, lang)
        )

    return (
        "\n".join(new_lines) + "\n",
        lang,
        header_variant,
        footer_variant,
        original_comments_kept,
        original_comments_removed,
    )


# ============================================================
# Ejecutar generación local Type I
# Objetivo:
# - type1_errors_df debe quedar vacío.
# - originales que no pasan sintaxis se generan con identity fallback
#   y se registran como advertencias, NO como errores.
# ============================================================

type1_rows = []
type1_errors = []
type1_warnings = []

for dataset_name, paths in DATASETS.items():
    jsfiles_dir = Path(paths["js_files"])
    clones_dir = Path(paths["clones"])
    clones_dir.mkdir(parents=True, exist_ok=True)

    originals = sorted(jsfiles_dir.glob("*.js"))
    print(f"{dataset_name}: {len(originals)} originales para Type I local")

    for orig_path in originals:
        file_id = orig_path.stem

        if ONLY_FILE_IDS is not None and file_id not in set(map(str, ONLY_FILE_IDS)):
            continue

        clone_path = clones_dir / f"{file_id}_clone_1.js"
        original_code = read_code(orig_path)

        original_syntax_ok, original_syntax_reason, original_syntax_mode = syntax_check_auto_mode(
            original_code,
            file_id=file_id,
            prefix="type1_original",
            report_dir=REPORT_DIR,
        )

        # Si el original no pasa sintaxis, NO lo tratamos como error del generador.
        # Generamos identity fallback para conservar el par.
        if not original_syntax_ok:
            (
                clone_code,
                clone_language,
                header_variant,
                footer_variant,
                original_comments_kept,
                original_comments_removed,
            ) = make_type1_clone_identity_fallback(
                original_code,
                dataset_name=dataset_name,
                file_id=file_id,
            )

            token_ok, token_reason = safe_type1_tokens_equivalent(original_code, clone_code)

            clone_path.write_text(clone_code, encoding="utf-8")

            type1_rows.append({
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_1.js",
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": 1,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "type1_identity_original_syntax_unchecked",
                "syntax_ok": False,
                "type1_token_ok": token_ok,
                "comment_language": clone_language,
                "comment_language_name": LANGUAGE_NAMES[clone_language],
                "header_variant": header_variant,
                "footer_variant": footer_variant,
                "original_comments_kept": original_comments_kept,
                "original_comments_removed": original_comments_removed,
                "fallback_used": "identity_original_syntax_failed",
                "original_syntax_ok": False,
                "original_syntax_mode": original_syntax_mode,
            })

            type1_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "original_syntax_failed_identity_written",
                "reason": original_syntax_reason,
                "token_check_reason": token_reason,
                "syntax_mode": original_syntax_mode,
            })

            continue

        (
            clone_code,
            clone_language,
            header_variant,
            footer_variant,
            original_comments_kept,
            original_comments_removed,
        ) = make_type1_clone_locally(
            original_code,
            dataset_name=dataset_name,
            file_id=file_id,
        )

        token_ok, token_reason = safe_type1_tokens_equivalent(original_code, clone_code)

        # Si el clon rico falla equivalencia de tokens, usar fallback identity.
        if not token_ok:
            type1_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "rich_clone_token_mismatch_identity_fallback",
                "reason": token_reason or "Rich Type I clone produced token mismatch.",
                "syntax_mode": original_syntax_mode,
            })

            (
                clone_code,
                clone_language,
                header_variant,
                footer_variant,
                original_comments_kept,
                original_comments_removed,
            ) = make_type1_clone_identity_fallback(
                original_code,
                dataset_name=dataset_name,
                file_id=file_id,
            )

            token_ok, token_reason = safe_type1_tokens_equivalent(original_code, clone_code)

        # Si incluso identity falla, no lo metemos a errores; lo marcamos como warning.
        if not token_ok:
            type1_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "identity_token_check_failed",
                "reason": token_reason or "Even identity fallback failed token equivalence.",
                "syntax_mode": original_syntax_mode,
            })

        syntax_ok, syntax_reason, clone_syntax_mode = syntax_check_auto_mode(
            clone_code,
            file_id=file_id,
            prefix="type1_local",
            report_dir=REPORT_DIR,
        )

        # Si el clon rico quedó inválido aunque el original era válido,
        # volver a identity fallback. Ese identity debería pasar igual que el original.
        if not syntax_ok:
            type1_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "rich_clone_syntax_failed_identity_fallback",
                "reason": syntax_reason,
                "syntax_mode": clone_syntax_mode,
            })

            (
                clone_code,
                clone_language,
                header_variant,
                footer_variant,
                original_comments_kept,
                original_comments_removed,
            ) = make_type1_clone_identity_fallback(
                original_code,
                dataset_name=dataset_name,
                file_id=file_id,
            )

            token_ok, token_reason = safe_type1_tokens_equivalent(original_code, clone_code)

            syntax_ok, syntax_reason, clone_syntax_mode = syntax_check_auto_mode(
                clone_code,
                file_id=file_id,
                prefix="type1_local_identity",
                report_dir=REPORT_DIR,
            )

            if not syntax_ok:
                type1_warnings.append({
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "original_path": str(orig_path),
                    "clone_path": str(clone_path),
                    "status": "identity_syntax_check_failed_after_original_valid",
                    "reason": syntax_reason,
                    "syntax_mode": clone_syntax_mode,
                })

        clone_path.write_text(clone_code, encoding="utf-8")

        row_status = "type1_generated_locally"
        if header_variant == "identity_fallback":
            row_status = "type1_identity_fallback"

        type1_rows.append({
            "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_1.js",
            "dataset": dataset_name,
            "file_id": file_id,
            "clone_type": 1,
            "original_path": str(orig_path),
            "clone_path": str(clone_path),
            "status": row_status,
            "syntax_ok": syntax_ok,
            "type1_token_ok": token_ok,
            "comment_language": clone_language,
            "comment_language_name": LANGUAGE_NAMES[clone_language],
            "header_variant": header_variant,
            "footer_variant": footer_variant,
            "original_comments_kept": original_comments_kept,
            "original_comments_removed": original_comments_removed,
            "fallback_used": header_variant == "identity_fallback",
            "original_syntax_ok": original_syntax_ok,
            "original_syntax_mode": original_syntax_mode,
            "clone_syntax_mode": clone_syntax_mode,
        })


type1_local_df = pd.DataFrame(type1_rows)
type1_errors_df = pd.DataFrame(type1_errors)
type1_warnings_df = pd.DataFrame(type1_warnings)

type1_local_csv = REPORTS_DIR / "type1_generated_locally.csv"
type1_errors_csv = REPORTS_DIR / "type1_local_errors.csv"
type1_warnings_csv = REPORTS_DIR / "type1_warnings.csv"

type1_local_df.to_csv(type1_local_csv, index=False)
type1_errors_df.to_csv(type1_errors_csv, index=False)
type1_warnings_df.to_csv(type1_warnings_csv, index=False)

print("Type I generados localmente:", len(type1_rows))
print("Type I errores del generador:", len(type1_errors))
print("Type I warnings / fallbacks:", len(type1_warnings))
print("CSV Type I local:", type1_local_csv)
print("CSV Type I errores:", type1_errors_csv)
print("CSV Type I warnings:", type1_warnings_csv)

if len(type1_errors_df):
    print("\nDistribución de errores:")
    print(type1_errors_df["status"].value_counts())
else:
    print("\nDistribución de errores:")
    print("Sin errores del generador.")

if len(type1_warnings_df):
    print("\nDistribución de warnings / fallbacks:")
    print(type1_warnings_df["status"].value_counts())

    print("\nRazones más comunes:")
    display(type1_warnings_df["reason"].value_counts().head(30))

if len(type1_local_df):
    print("\nDistribución de status:")
    display(type1_local_df["status"].value_counts())

    print("\nDistribución de idiomas:")
    display(type1_local_df["comment_language_name"].value_counts())

Componentes de comentarios OK: 100 variantes por elemento e idioma
Construyendo bancos de comentarios simples...
fr (francés): 20000 frases únicas
de (alemán): 20000 frases únicas
es (español): 20000 frases únicas
en (inglés): 20000 frases únicas
ko (coreano): 20000 frases únicas
ja (japonés): 20000 frases únicas
hi (hindi): 20000 frases únicas
zh (chino): 20000 frases únicas
pt (portugués): 20000 frases únicas
ru (ruso): 20000 frases únicas
la (latín): 20000 frases únicas
Bancos simples OK
HEADER_STYLES total: 40
dataset solutions: 11001 originales para Type I local
Type I generados localmente: 11001
Type I errores del generador: 0
Type I warnings / fallbacks: 19
CSV Type I local: generation/run_20260628_230814/reports/type1_generated_locally.csv
CSV Type I errores: generation/run_20260628_230814/reports/type1_local_errors.csv
CSV Type I warnings: generation/run_20260628_230814/reports/type1_warnings.csv

Distribución de errores:
Sin errores del generador.

Distribución de warnings / 

reason
Rich Type I clone produced token mismatch.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             15
(node:3129235) Warning: Failed to load the ES module: /home/rodrigo/Documentos/Github/Tacco-JS/dataset/generation/run_20260628_230814/_type1_original_933.cjs. Make sure to set "type": "module" in the nearest package.json file or use the .mjs extension.\n(Use `node --trace-warnings ...` to show where the warnin


Distribución de status:


status
type1_generated_locally                     10965
type1_identity_fallback                        32
type1_identity_original_syntax_unchecked        4
Name: count, dtype: int64


Distribución de idiomas:


comment_language_name
francés      1048
latín        1027
japonés      1004
ruso          999
portugués     997
alemán        996
inglés        993
hindi         989
coreano       989
chino         981
español       978
Name: count, dtype: int64

In [6]:
# Cell auxiliar — Validación de sintaxis JS para Prettier / Type II

import re
import subprocess
from pathlib import Path

ESM_IMPORT_RE = re.compile(
    r"^\s*import\s+(?:[\w*{]|['\"])",
    re.MULTILINE,
)

ESM_EXPORT_RE = re.compile(
    r"^\s*export\s+(?:default|const|let|var|function|class|\{|\*)",
    re.MULTILINE,
)


def looks_like_esm(code):
    """
    Detecta si el archivo parece ES Module.
    """
    return bool(ESM_IMPORT_RE.search(code) or ESM_EXPORT_RE.search(code))


def syntax_check_text(code, temp_path):
    """
    Valida sintaxis JS usando node --check.

    Devuelve:
    - ok: bool
    - reason: str
    """
    temp_path = Path(temp_path)
    temp_path.parent.mkdir(parents=True, exist_ok=True)

    safe_code = str(code).encode("utf-8", errors="backslashreplace").decode("utf-8")
    temp_path.write_text(safe_code, encoding="utf-8")

    try:
        result = subprocess.run(
            ["node", "--check", str(temp_path)],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=30,
        )

        if result.returncode == 0:
            return True, "ok"

        reason = result.stderr.strip() or result.stdout.strip() or "node --check failed"
        return False, reason

    except FileNotFoundError:
        return False, "node command not found. Instala nodejs/npm o revisa tu PATH."

    except subprocess.TimeoutExpired:
        return False, "node --check timeout"

    finally:
        try:
            temp_path.unlink()
        except OSError:
            pass


def syntax_check_auto_mode(code, file_id, prefix, report_dir):
    """
    Intenta validar el código como .mjs o .cjs según parezca módulo o script.

    Devuelve:
    - ok
    - reason
    - extension usada
    """
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)

    modes = [".mjs", ".cjs"] if looks_like_esm(code) else [".cjs", ".mjs"]

    last_reason = ""

    safe_file_id = str(file_id).replace("/", "_").replace("\\", "_")

    for ext in modes:
        temp_path = report_dir / f"_{prefix}_{safe_file_id}{ext}"
        ok, reason = syntax_check_text(code, temp_path=temp_path)

        if ok:
            return True, reason, ext

        last_reason = reason

    return False, last_reason, modes[-1]


print("Funciones de validación JS cargadas correctamente.")

Funciones de validación JS cargadas correctamente.


In [ ]:
# Cell 8.X — Reformatear subconjunto de Type II con Prettier

# Objetivo:
# - Tomar algunos *_clone_2.js ya generados.
# - Aplicar Prettier solo a un subconjunto controlado.
# - No hacerlo manualmente archivo por archivo.
# - Validar sintaxis antes y después.
# - Guardar backup antes de sobrescribir.
# - Guardar CSV de auditoría.

import random
import hashlib
import shutil
import subprocess
import pandas as pd
from pathlib import Path

TYPE2_PRETTIER_SEED = 44

# Recomendado: 0.20 a 0.40
TYPE2_PRETTIER_PROB = 0.30

# Si True, sobrescribe el clone_2 original.
# Si False, crea un archivo separado: *_clone_2_prettier.js
TYPE2_PRETTIER_IN_PLACE = True

# Recomendado dejar True
TYPE2_PRETTIER_MAKE_BACKUP = True

# Si True, intenta validar que Prettier no cambió tokens ejecutables.
# OJO: puede fallar si Prettier cambia comillas, paréntesis o trailing commas.
# Por eso lo dejo False por defecto.
TYPE2_PRETTIER_STRICT_TOKEN_CHECK = False

# Si no tienes prettier local, intenta usar npx.
TYPE2_PRETTIER_USE_NPX_FALLBACK = True

# Versión fija para reproducibilidad si usa npx.
TYPE2_PRETTIER_NPX_SPEC = "prettier@3.3.3"

TYPE2_PRETTIER_BACKUP_DIRNAME = "_type2_prettier_backups"


def stable_rng_for_type2_prettier(dataset_name, file_id, seed=TYPE2_PRETTIER_SEED):
    key = f"type2_prettier|{seed}|{dataset_name}|{file_id}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    value = int(digest[:16], 16)
    return random.Random(value)


def should_apply_prettier_to_type2(dataset_name, file_id):
    rng = stable_rng_for_type2_prettier(dataset_name, file_id)
    return rng.random() < TYPE2_PRETTIER_PROB


def read_code_safe(path):
    if "read_code" in globals():
        return read_code(path)
    return Path(path).read_text(encoding="utf-8", errors="surrogateescape")


def write_code_safe(path, code):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    safe = str(code).encode("utf-8", errors="backslashreplace").decode("utf-8")
    path.write_text(safe, encoding="utf-8")


def syntax_check_type2_prettier(code, file_id, prefix, report_dir):
    if "syntax_check_auto_mode" not in globals():
        raise RuntimeError(
            "No existe syntax_check_auto_mode en el notebook. "
            "Ejecuta primero tus celdas base de validación."
        )

    return syntax_check_auto_mode(
        code,
        file_id=file_id,
        prefix=prefix,
        report_dir=report_dir,
    )


def safe_token_equivalence_before_after(before_code, after_code):
    """
    Validación estricta opcional.
    No es obligatoria porque Prettier puede cambiar comillas o paréntesis
    sin cambiar la semántica.
    """
    if "safe_type1_tokens_equivalent" in globals():
        return safe_type1_tokens_equivalent(before_code, after_code)

    if "type1_tokens_equivalent" in globals():
        try:
            return bool(type1_tokens_equivalent(before_code, after_code)), ""
        except Exception as exc:
            return False, repr(exc)

    return None, "No token-equivalence function available."


def resolve_prettier_command():
    """
    Usa prettier local/global si existe.
    Si no, usa npx prettier@versión.
    """
    prettier_bin = shutil.which("prettier")

    if prettier_bin:
        return [prettier_bin]

    if TYPE2_PRETTIER_USE_NPX_FALLBACK and shutil.which("npx"):
        return ["npx", "--yes", TYPE2_PRETTIER_NPX_SPEC]

    raise RuntimeError(
        "No encontré prettier ni npx. Instala Prettier con:\n"
        "!npm install -D prettier --legacy-peer-deps"
    )


def format_with_prettier(code, file_path):
    """
    Formatea usando stdin para no modificar el archivo hasta validar.
    """
    base_cmd = resolve_prettier_command()

    cmd = base_cmd + [
        "--stdin-filepath",
        str(file_path),
        "--parser",
        "babel",
    ]

    result = subprocess.run(
        cmd,
        input=code,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=60,
    )

    if result.returncode != 0:
        return False, code, result.stderr.strip()

    formatted = result.stdout

    if not formatted.endswith("\n"):
        formatted += "\n"

    return True, formatted, ""


def backup_type2_file(clone_path, clones_dir, dataset_name):
    backup_dir = clones_dir / TYPE2_PRETTIER_BACKUP_DIRNAME
    backup_dir.mkdir(parents=True, exist_ok=True)

    backup_path = backup_dir / clone_path.name

    # No sobrescribir un backup previo.
    if backup_path.exists():
        stem = clone_path.stem
        suffix = clone_path.suffix
        digest = hashlib.sha256(
            f"{dataset_name}|{clone_path}|{TYPE2_PRETTIER_SEED}".encode("utf-8")
        ).hexdigest()[:8]
        backup_path = backup_dir / f"{stem}_{digest}{suffix}"

    shutil.copy2(clone_path, backup_path)
    return backup_path


# ============================================================
# Ejecutar Prettier sobre algunos Type II
# ============================================================

type2_prettier_rows = []
type2_prettier_warnings = []

for dataset_name, paths in DATASETS.items():
    clones_dir = Path(paths["clones"])
    clone2_files = sorted(clones_dir.glob("*_clone_2.js"))

    print(f"{dataset_name}: {len(clone2_files)} Type II encontrados")

    for clone_path in clone2_files:
        if len(type2_prettier_rows) % 250 == 0:
            print("Procesados:", len(type2_prettier_rows))
        file_id = clone_path.name.replace("_clone_2.js", "")

        if ONLY_FILE_IDS is not None and file_id not in set(map(str, ONLY_FILE_IDS)):
            continue

        selected = should_apply_prettier_to_type2(dataset_name, file_id)

        if not selected:
            type2_prettier_rows.append({
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_2.js",
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": 2,
                "clone_path": str(clone_path),
                "output_path": str(clone_path),
                "status": "skipped_probability",
                "prettier_applied": False,
                "syntax_before_ok": "",
                "syntax_after_ok": "",
                "token_equivalence_checked": False,
                "token_equivalence_ok": "",
                "backup_path": "",
                "reason": "",
            })
            continue

        original_clone2_code = read_code_safe(clone_path)

        syntax_before_ok, syntax_before_reason, syntax_before_mode = syntax_check_type2_prettier(
            original_clone2_code,
            file_id=file_id,
            prefix="type2_prettier_before",
            report_dir=REPORT_DIR,
        )

        if not syntax_before_ok:
            type2_prettier_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_path": str(clone_path),
                "status": "syntax_before_failed_skipped",
                "reason": syntax_before_reason,
                "syntax_mode": syntax_before_mode,
            })

            type2_prettier_rows.append({
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_2.js",
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": 2,
                "clone_path": str(clone_path),
                "output_path": str(clone_path),
                "status": "syntax_before_failed_skipped",
                "prettier_applied": False,
                "syntax_before_ok": False,
                "syntax_before_mode": syntax_before_mode,
                "syntax_after_ok": "",
                "syntax_after_mode": "",
                "token_equivalence_checked": False,
                "token_equivalence_ok": "",
                "backup_path": "",
                "reason": syntax_before_reason,
            })
            continue

        prettier_ok, formatted_code, prettier_reason = format_with_prettier(
            original_clone2_code,
            clone_path,
        )

        if not prettier_ok:
            type2_prettier_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_path": str(clone_path),
                "status": "prettier_failed_skipped",
                "reason": prettier_reason,
                "syntax_mode": syntax_before_mode,
            })

            type2_prettier_rows.append({
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_2.js",
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": 2,
                "clone_path": str(clone_path),
                "output_path": str(clone_path),
                "status": "prettier_failed_skipped",
                "prettier_applied": False,
                "syntax_before_ok": True,
                "syntax_before_mode": syntax_before_mode,
                "syntax_after_ok": "",
                "syntax_after_mode": "",
                "token_equivalence_checked": False,
                "token_equivalence_ok": "",
                "backup_path": "",
                "reason": prettier_reason,
            })
            continue

        syntax_after_ok, syntax_after_reason, syntax_after_mode = syntax_check_type2_prettier(
            formatted_code,
            file_id=file_id,
            prefix="type2_prettier_after",
            report_dir=REPORT_DIR,
        )

        if not syntax_after_ok:
            type2_prettier_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_path": str(clone_path),
                "status": "syntax_after_failed_skipped",
                "reason": syntax_after_reason,
                "syntax_mode": syntax_after_mode,
            })

            type2_prettier_rows.append({
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_2.js",
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": 2,
                "clone_path": str(clone_path),
                "output_path": str(clone_path),
                "status": "syntax_after_failed_skipped",
                "prettier_applied": False,
                "syntax_before_ok": True,
                "syntax_before_mode": syntax_before_mode,
                "syntax_after_ok": False,
                "syntax_after_mode": syntax_after_mode,
                "token_equivalence_checked": False,
                "token_equivalence_ok": "",
                "backup_path": "",
                "reason": syntax_after_reason,
            })
            continue

        token_equivalence_checked = False
        token_equivalence_ok = ""

        if TYPE2_PRETTIER_STRICT_TOKEN_CHECK:
            token_ok, token_reason = safe_token_equivalence_before_after(
                original_clone2_code,
                formatted_code,
            )

            token_equivalence_checked = True
            token_equivalence_ok = token_ok

            if token_ok is False:
                type2_prettier_warnings.append({
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "clone_path": str(clone_path),
                    "status": "token_equivalence_failed_skipped",
                    "reason": token_reason,
                    "syntax_mode": syntax_after_mode,
                })

                type2_prettier_rows.append({
                    "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_2.js",
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "clone_type": 2,
                    "clone_path": str(clone_path),
                    "output_path": str(clone_path),
                    "status": "token_equivalence_failed_skipped",
                    "prettier_applied": False,
                    "syntax_before_ok": True,
                    "syntax_before_mode": syntax_before_mode,
                    "syntax_after_ok": True,
                    "syntax_after_mode": syntax_after_mode,
                    "token_equivalence_checked": True,
                    "token_equivalence_ok": False,
                    "backup_path": "",
                    "reason": token_reason,
                })
                continue

        backup_path = ""

        if TYPE2_PRETTIER_IN_PLACE:
            output_path = clone_path

            if TYPE2_PRETTIER_MAKE_BACKUP:
                backup_path = backup_type2_file(clone_path, clones_dir, dataset_name)

        else:
            output_path = clones_dir / f"{file_id}_clone_2_prettier.js"

        write_code_safe(output_path, formatted_code)

        type2_prettier_rows.append({
            "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_2.js",
            "dataset": dataset_name,
            "file_id": file_id,
            "clone_type": 2,
            "clone_path": str(clone_path),
            "output_path": str(output_path),
            "status": "prettier_applied",
            "prettier_applied": True,
            "syntax_before_ok": True,
            "syntax_before_mode": syntax_before_mode,
            "syntax_after_ok": True,
            "syntax_after_mode": syntax_after_mode,
            "token_equivalence_checked": token_equivalence_checked,
            "token_equivalence_ok": token_equivalence_ok,
            "backup_path": str(backup_path) if backup_path else "",
            "reason": "",
            "changed_text": original_clone2_code != formatted_code,
            "original_chars": len(original_clone2_code),
            "formatted_chars": len(formatted_code),
        })


type2_prettier_df = pd.DataFrame(type2_prettier_rows)
type2_prettier_warnings_df = pd.DataFrame(type2_prettier_warnings)

type2_prettier_csv = REPORTS_DIR / "type2_prettier_reformatted.csv"
type2_prettier_warnings_csv = REPORTS_DIR / "type2_prettier_warnings.csv"

type2_prettier_df.to_csv(type2_prettier_csv, index=False)
type2_prettier_warnings_df.to_csv(type2_prettier_warnings_csv, index=False)

print("Type II revisados:", len(type2_prettier_rows))
print("Type II con Prettier aplicado:", int(type2_prettier_df["prettier_applied"].fillna(False).sum()) if len(type2_prettier_df) else 0)
print("Warnings / skips:", len(type2_prettier_warnings_df))
print("CSV Type II Prettier:", type2_prettier_csv)
print("CSV warnings:", type2_prettier_warnings_csv)

if len(type2_prettier_df):
    print("\nDistribución de status:")
    display(type2_prettier_df["status"].value_counts())

    if "changed_text" in type2_prettier_df.columns:
        print("\nArchivos donde Prettier cambió texto:")
        display(type2_prettier_df["changed_text"].value_counts(dropna=False))

if len(type2_prettier_warnings_df):
    print("\nDistribución de warnings:")
    display(type2_prettier_warnings_df["status"].value_counts())

    if "reason" in type2_prettier_warnings_df.columns:
        print("\nRazones más comunes:")
        display(type2_prettier_warnings_df["reason"].value_counts().head(30))

dataset solutions: 11001 Type II encontrados
Procesados: 0


RuntimeError: No encontré Prettier local. Instálalo primero con:
!npm install -D prettier@3.3.3 --ignore-scripts --legacy-peer-deps

In [ ]:
# Cell 8.7 — Regenerar Type III localmente, sin LLM, con near-miss clones variados

# Reglas Type III:
# - Debe preservar la semántica del original.
# - Puede agregar statements, bloques, funciones auxiliares no usadas o ramas inalcanzables.
# - No debe usar un patrón único repetido como if(false) console.log(...).
# - No debe incluir strings obvios como "clone", "type3", "dead_clone_branch".
# - Se valida con:
#   1. node --check / syntax_check_auto_mode
#   2. fallback si el clon generado falla sintaxis
# - Recomendado:
#   - Si existe clone_2, usarlo como base para Type III.
#   - Si no existe, usar el original.

import random
import hashlib
import json
import re
import pandas as pd
from pathlib import Path

from tqdm import tqdm

try:
    from tree_sitter import Parser
    import tree_sitter_javascript
except ImportError as exc:
    raise RuntimeError(
        "Faltan dependencias. Ejecuta:\n"
        "!python -m pip install tree-sitter tree-sitter-javascript tqdm"
    ) from exc


TYPE3_LOCAL_SEED = 43
TYPE3_MIN_EDITS = 2
TYPE3_MAX_EDITS = 6
TYPE3_MAX_ATTEMPTS = 20

# Si existe file_id_clone_2.js, lo usa como base y encima agrega cambios Type III.
# Esto hace que Type III sea más realista: Type II + near-miss changes.
TYPE3_USE_TYPE2_AS_BASE_IF_EXISTS = True

# Si falla la generación rica, escribe fallback seguro.
TYPE3_USE_IDENTITY_FALLBACK_ON_FAILURE = True


# ============================================================
# Parser JS
# ============================================================

def get_tree_sitter_js_language():
    """
    Compatible con varias versiones de tree-sitter / tree-sitter-javascript.
    """
    lang = tree_sitter_javascript.language()

    try:
        from tree_sitter import Language
        return Language(lang)
    except TypeError:
        return lang


def build_js_parser():
    parser = Parser()
    language = get_tree_sitter_js_language()

    if hasattr(parser, "set_language"):
        parser.set_language(language)
    else:
        parser.language = language

    return parser


JS_PARSER = build_js_parser()


# ============================================================
# Utilidades generales
# ============================================================

def stable_rng_for_type3(dataset_name, file_id, seed=TYPE3_LOCAL_SEED):
    key = f"type3|{seed}|{dataset_name}|{file_id}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    value = int(digest[:16], 16)
    return random.Random(value)


def normalize_final_newline_type3(code):
    return code if code.endswith("\n") else code + "\n"


def write_code_safe(path, code):
    """
    Escribe JS evitando errores por surrogates raros.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    safe = str(code).encode("utf-8", errors="backslashreplace").decode("utf-8")
    path.write_text(safe, encoding="utf-8")


def syntax_check_type3(code, file_id, prefix, report_dir):
    """
    Usa syntax_check_auto_mode si ya existe en tu notebook.
    Si no existe, lanza error claro.
    """
    if "syntax_check_auto_mode" not in globals():
        raise RuntimeError(
            "No existe syntax_check_auto_mode en el notebook. "
            "Ejecuta primero las celdas base que usas para Type I."
        )

    return syntax_check_auto_mode(
        code,
        file_id=file_id,
        prefix=prefix,
        report_dir=report_dir,
    )


def make_type3_identity_fallback(code):
    """
    Fallback conservador. No es un Type III rico, pero evita perder el par.
    """
    return normalize_final_newline_type3(code), "identity_fallback", []


def random_identifier_type3(rng, used_names):
    """
    Genera identificadores no obvios.
    Evita palabras como clone/type/dead.
    """
    alphabet = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    chars = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

    for _ in range(1000):
        suffix = "".join(rng.choice(chars) for _ in range(rng.randint(8, 14)))
        name = f"_{rng.choice(alphabet)}{suffix}"

        lowered = name.lower()
        if "clone" in lowered or "type" in lowered or "dead" in lowered:
            continue

        if name not in used_names:
            used_names.add(name)
            return name

    digest = hashlib.sha256(str(rng.random()).encode()).hexdigest()[:12]
    name = f"_v{digest}"
    used_names.add(name)
    return name


def random_type3_string(rng):
    words = [
        "alpha", "buffer", "state", "cache", "entry", "index",
        "packet", "delta", "cursor", "frame", "node", "value",
        "segment", "scope", "token", "unit", "route", "field",
        "signal", "layer", "matrix", "window", "branch", "table",
    ]
    return rng.choice(words) + "_" + "".join(rng.choice("0123456789abcdef") for _ in range(8))


def indent_for_offset(text, offset):
    prefix = text[:offset]
    last_newline = prefix.rfind("\n")
    line_start = 0 if last_newline == -1 else last_newline + 1
    line_prefix = prefix[line_start:]

    match = re.match(r"[ \t]*", line_prefix)
    return match.group(0) if match else ""


# ============================================================
# Recolección de puntos de inserción mediante AST
# ============================================================

TYPE3_CONTAINER_TYPES = {
    "program",
    "statement_block",
}

TYPE3_STATEMENT_TYPES = {
    "expression_statement",
    "lexical_declaration",
    "variable_declaration",
    "function_declaration",
    "return_statement",
    "if_statement",
    "for_statement",
    "for_in_statement",
    "for_of_statement",
    "while_statement",
    "do_statement",
    "switch_statement",
    "try_statement",
    "throw_statement",
    "break_statement",
    "continue_statement",
    "debugger_statement",
    "import_statement",
    "export_statement",
    "class_declaration",
}


def is_type3_statement_like(node_type):
    return node_type in TYPE3_STATEMENT_TYPES or node_type.endswith("_statement")


def collect_type3_insertion_points(code):
    """
    Devuelve offsets seguros para insertar statements.
    Solo inserta dentro de:
    - program
    - statement_block

    Evita meter código dentro de strings, comentarios o expresiones.
    """
    source = code.encode("utf-8", errors="surrogatepass")
    tree = JS_PARSER.parse(source)
    root = tree.root_node

    points = []

    def visit(node):
        if node.type in TYPE3_CONTAINER_TYPES:
            named_children = [
                child for child in node.named_children
                if is_type3_statement_like(child.type)
            ]

            for child in named_children:
                # Antes del statement.
                points.append({
                    "offset": child.start_byte,
                    "line": child.start_point[0] + 1,
                    "container": node.type,
                    "position": "before",
                })

                # Después del statement.
                points.append({
                    "offset": child.end_byte,
                    "line": child.end_point[0] + 1,
                    "container": node.type,
                    "position": "after",
                })

            # Bloques vacíos: insertar después de {
            if node.type == "statement_block" and not named_children:
                open_brace = source.find(b"{", node.start_byte, node.end_byte)
                if open_brace != -1:
                    points.append({
                        "offset": open_brace + 1,
                        "line": node.start_point[0] + 1,
                        "container": node.type,
                        "position": "inside_empty_block",
                    })

        for child in node.named_children:
            visit(child)

    visit(root)

    # Evitar inserción antes del shebang.
    filtered = []
    for p in points:
        if code.startswith("#!") and p["offset"] == 0:
            continue
        filtered.append(p)

    # Deduplicar offsets.
    unique = {}
    for p in filtered:
        unique[p["offset"]] = p

    return list(unique.values())


# ============================================================
# Estrategias Type III variadas
# ============================================================

TYPE3_STRATEGIES = [
    "unused_const_string",
    "unused_const_number",
    "unused_const_boolean_expr",
    "void_literal",
    "empty_local_block",
    "zero_iteration_for",
    "zero_iteration_while",
    "unreachable_numeric_branch",
    "unreachable_string_branch",
    "unused_arrow_function",
    "unused_function_expression",
    "try_finally_noop",
    "switch_unreachable_case",
    "nested_noop_block",
    "array_literal_unused",
    "object_literal_unused",
]


def make_type3_snippet(strategy, rng, indent, used_names):
    a = random_identifier_type3(rng, used_names)
    b = random_identifier_type3(rng, used_names)
    c = random_identifier_type3(rng, used_names)

    n1 = rng.randint(1, 9999)
    n2 = n1 + rng.randint(1, 9999)
    s1 = random_type3_string(rng)
    s2 = random_type3_string(rng)

    # Evitar console.log, clone, type3, dead.
    # Todos estos snippets son no-op o inalcanzables.

    if strategy == "unused_const_string":
        return f"\n{indent}const {a} = {json.dumps(s1)};\n"

    if strategy == "unused_const_number":
        return f"\n{indent}const {a} = {n1} + {n2} - {n2};\n"

    if strategy == "unused_const_boolean_expr":
        return f"\n{indent}const {a} = ({n1} < {n2}) && true;\n"

    if strategy == "void_literal":
        return f"\n{indent}void {json.dumps(s1)};\n"

    if strategy == "empty_local_block":
        return (
            f"\n{indent}{{\n"
            f"{indent}  const {a} = {n1};\n"
            f"{indent}  void {a};\n"
            f"{indent}}}\n"
        )

    if strategy == "zero_iteration_for":
        return (
            f"\n{indent}for (let {a} = 0; {a} < 0; {a}++) {{\n"
            f"{indent}  void {a};\n"
            f"{indent}}}\n"
        )

    if strategy == "zero_iteration_while":
        return (
            f"\n{indent}while (false) {{\n"
            f"{indent}  const {a} = {json.dumps(s1)};\n"
            f"{indent}  void {a};\n"
            f"{indent}}}\n"
        )

    if strategy == "unreachable_numeric_branch":
        return (
            f"\n{indent}if ({n1} > {n2}) {{\n"
            f"{indent}  const {a} = {n1} * {n2};\n"
            f"{indent}  void {a};\n"
            f"{indent}}}\n"
        )

    if strategy == "unreachable_string_branch":
        return (
            f"\n{indent}if ({json.dumps(s1)}.length < 0) {{\n"
            f"{indent}  const {a} = {json.dumps(s2)};\n"
            f"{indent}  void {a};\n"
            f"{indent}}}\n"
        )

    if strategy == "unused_arrow_function":
        return f"\n{indent}const {a} = ({b}) => {b};\n"

    if strategy == "unused_function_expression":
        return (
            f"\n{indent}const {a} = function({b}) {{\n"
            f"{indent}  return {b};\n"
            f"{indent}}};\n"
        )

    if strategy == "try_finally_noop":
        return (
            f"\n{indent}try {{\n"
            f"{indent}  void {n1};\n"
            f"{indent}}} finally {{\n"
            f"{indent}}}\n"
        )

    if strategy == "switch_unreachable_case":
        return (
            f"\n{indent}switch ({n1}) {{\n"
            f"{indent}  case {n2}:\n"
            f"{indent}    void {json.dumps(s1)};\n"
            f"{indent}    break;\n"
            f"{indent}}}\n"
        )

    if strategy == "nested_noop_block":
        return (
            f"\n{indent}{{\n"
            f"{indent}  {{\n"
            f"{indent}    const {a} = {json.dumps(s1)};\n"
            f"{indent}    void {a};\n"
            f"{indent}  }}\n"
            f"{indent}}}\n"
        )

    if strategy == "array_literal_unused":
        return f"\n{indent}const {a} = [{n1}, {n2}, {n1}];\n"

    if strategy == "object_literal_unused":
        return f"\n{indent}const {a} = {{ {json.dumps(s1)}: {n1}, {json.dumps(s2)}: {n2} }};\n"

    raise ValueError(f"Estrategia Type III desconocida: {strategy}")


def apply_type3_edits(code, rng):
    points = collect_type3_insertion_points(code)

    if not points:
        raise ValueError("No valid Type III insertion points found.")

    used_names = set()
    edit_count = rng.randint(TYPE3_MIN_EDITS, TYPE3_MAX_EDITS)
    selected_points = rng.sample(points, k=min(edit_count, len(points)))

    insertions = []
    applied = []

    for point in selected_points:
        strategy = rng.choice(TYPE3_STRATEGIES)
        indent = indent_for_offset(code, point["offset"])
        snippet = make_type3_snippet(strategy, rng, indent, used_names)

        snippet_hash = hashlib.sha256(
            snippet.encode("utf-8", errors="replace")
        ).hexdigest()[:12]

        insertions.append((point["offset"], snippet))
        applied.append({
            "strategy": strategy,
            "line": point["line"],
            "offset": point["offset"],
            "container": point["container"],
            "position": point["position"],
            "snippet_hash": snippet_hash,
        })

    # Aplicar de derecha a izquierda para que no cambien los offsets.
    output = code
    for offset, snippet in sorted(insertions, key=lambda item: item[0], reverse=True):
        output = output[:offset] + snippet + output[offset:]

    return normalize_final_newline_type3(output), applied


def make_type3_clone_locally(code, dataset_name="", file_id=""):
    """
    Genera un Type III local más completo:
    - Varias inserciones no-op / unreachable.
    - Puntos de inserción basados en AST.
    - Validación posterior con node --check.
    """
    rng = stable_rng_for_type3(dataset_name, file_id)

    for attempt in range(1, TYPE3_MAX_ATTEMPTS + 1):
        try:
            clone_code, applied = apply_type3_edits(code, rng)
            return clone_code, "type3_generated_locally", applied, attempt, ""
        except Exception as exc:
            last_error = repr(exc)

    if TYPE3_USE_IDENTITY_FALLBACK_ON_FAILURE:
        clone_code, status, applied = make_type3_identity_fallback(code)
        return clone_code, status, applied, TYPE3_MAX_ATTEMPTS, last_error

    raise RuntimeError(last_error)


# ============================================================
# Ejecutar generación local Type III
# ============================================================

type3_rows = []
type3_errors = []
type3_warnings = []

for dataset_name, paths in DATASETS.items():
    jsfiles_dir = Path(paths["js_files"])
    clones_dir = Path(paths["clones"])
    clones_dir.mkdir(parents=True, exist_ok=True)

    originals = sorted(jsfiles_dir.glob("*.js"))
    print(f"{dataset_name}: {len(originals)} originales para Type III local")

    for orig_path in tqdm(originals, desc=f"Type III {dataset_name}"):
        file_id = orig_path.stem

        if ONLY_FILE_IDS is not None and file_id not in set(map(str, ONLY_FILE_IDS)):
            continue

        clone_path = clones_dir / f"{file_id}_clone_3.js"

        original_code = read_code(orig_path)

        original_syntax_ok, original_syntax_reason, original_syntax_mode = syntax_check_type3(
            original_code,
            file_id=file_id,
            prefix="type3_original",
            report_dir=REPORT_DIR,
        )

        if not original_syntax_ok:
            clone_code, status, applied = make_type3_identity_fallback(original_code)
            write_code_safe(clone_path, clone_code)

            type3_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "original_syntax_failed_identity_written",
                "reason": original_syntax_reason,
                "syntax_mode": original_syntax_mode,
            })

            type3_rows.append({
                "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_3.js",
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": 3,
                "original_path": str(orig_path),
                "source_for_type3": str(orig_path),
                "clone_path": str(clone_path),
                "status": status,
                "syntax_ok": False,
                "original_syntax_ok": False,
                "original_syntax_mode": original_syntax_mode,
                "clone_syntax_mode": "",
                "edits_applied": 0,
                "strategies": "",
                "edit_lines": "",
                "snippet_hashes": "",
                "attempts": 0,
                "fallback_used": True,
            })

            continue

        # Base recomendada para Type III:
        # si existe Type II, generar Type III encima del Type II.
        source_for_type3_path = orig_path
        base_code = original_code
        base_kind = "original"

        possible_type2_path = clones_dir / f"{file_id}_clone_2.js"

        if TYPE3_USE_TYPE2_AS_BASE_IF_EXISTS and possible_type2_path.exists():
            candidate_type2_code = read_code(possible_type2_path)

            type2_syntax_ok, type2_syntax_reason, type2_syntax_mode = syntax_check_type3(
                candidate_type2_code,
                file_id=file_id,
                prefix="type3_base_type2",
                report_dir=REPORT_DIR,
            )

            if type2_syntax_ok:
                source_for_type3_path = possible_type2_path
                base_code = candidate_type2_code
                base_kind = "type2_clone"
            else:
                type3_warnings.append({
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "original_path": str(orig_path),
                    "clone_path": str(clone_path),
                    "status": "type2_base_syntax_failed_using_original",
                    "reason": type2_syntax_reason,
                    "syntax_mode": type2_syntax_mode,
                })

        try:
            clone_code, row_status, applied, attempts, generation_reason = make_type3_clone_locally(
                base_code,
                dataset_name=dataset_name,
                file_id=file_id,
            )
        except Exception as exc:
            generation_reason = repr(exc)
            clone_code, row_status, applied = make_type3_identity_fallback(base_code)
            attempts = TYPE3_MAX_ATTEMPTS

            type3_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "type3_generation_exception_identity_fallback",
                "reason": generation_reason,
                "syntax_mode": original_syntax_mode,
            })

        syntax_ok, syntax_reason, clone_syntax_mode = syntax_check_type3(
            clone_code,
            file_id=file_id,
            prefix="type3_local",
            report_dir=REPORT_DIR,
        )

        if not syntax_ok:
            type3_warnings.append({
                "dataset": dataset_name,
                "file_id": file_id,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "status": "type3_syntax_failed_identity_fallback",
                "reason": syntax_reason,
                "syntax_mode": clone_syntax_mode,
            })

            clone_code, row_status, applied = make_type3_identity_fallback(base_code)
            attempts = TYPE3_MAX_ATTEMPTS

            syntax_ok, syntax_reason, clone_syntax_mode = syntax_check_type3(
                clone_code,
                file_id=file_id,
                prefix="type3_local_identity",
                report_dir=REPORT_DIR,
            )

        write_code_safe(clone_path, clone_code)

        type3_rows.append({
            "id": f"{dataset_name.replace(' ', '_')}|{file_id}_clone_3.js",
            "dataset": dataset_name,
            "file_id": file_id,
            "clone_type": 3,
            "original_path": str(orig_path),
            "source_for_type3": str(source_for_type3_path),
            "source_for_type3_kind": base_kind,
            "clone_path": str(clone_path),
            "status": row_status,
            "syntax_ok": syntax_ok,
            "original_syntax_ok": original_syntax_ok,
            "original_syntax_mode": original_syntax_mode,
            "clone_syntax_mode": clone_syntax_mode,
            "edits_applied": len(applied),
            "strategies": "|".join(item["strategy"] for item in applied),
            "edit_lines": "|".join(str(item["line"]) for item in applied),
            "snippet_hashes": "|".join(item["snippet_hash"] for item in applied),
            "attempts": attempts,
            "fallback_used": row_status == "identity_fallback",
            "generation_reason": generation_reason,
        })


type3_local_df = pd.DataFrame(type3_rows)
type3_errors_df = pd.DataFrame(type3_errors)
type3_warnings_df = pd.DataFrame(type3_warnings)

type3_local_csv = REPORTS_DIR / "type3_generated_locally.csv"
type3_errors_csv = REPORTS_DIR / "type3_local_errors.csv"
type3_warnings_csv = REPORTS_DIR / "type3_warnings.csv"

type3_local_df.to_csv(type3_local_csv, index=False)
type3_errors_df.to_csv(type3_errors_csv, index=False)
type3_warnings_df.to_csv(type3_warnings_csv, index=False)

print("Type III generados localmente:", len(type3_rows))
print("Type III errores del generador:", len(type3_errors))
print("Type III warnings / fallbacks:", len(type3_warnings))
print("CSV Type III local:", type3_local_csv)
print("CSV Type III errores:", type3_errors_csv)
print("CSV Type III warnings:", type3_warnings_csv)

if len(type3_errors_df):
    print("\nDistribución de errores:")
    print(type3_errors_df["status"].value_counts())
else:
    print("\nDistribución de errores:")
    print("Sin errores del generador.")

if len(type3_warnings_df):
    print("\nDistribución de warnings / fallbacks:")
    print(type3_warnings_df["status"].value_counts())

    if "reason" in type3_warnings_df.columns:
        print("\nRazones más comunes:")
        display(type3_warnings_df["reason"].value_counts().head(30))

if len(type3_local_df):
    print("\nDistribución de status:")
    display(type3_local_df["status"].value_counts())

    print("\nBase usada para Type III:")
    display(type3_local_df["source_for_type3_kind"].value_counts())

    print("\nEstrategias más usadas:")
    all_strategies = []
    for value in type3_local_df["strategies"].dropna():
        all_strategies.extend([x for x in str(value).split("|") if x])
    display(pd.Series(all_strategies).value_counts().head(30))

    print("\nEdits por archivo:")
    display(type3_local_df["edits_applied"].value_counts().sort_index())

In [ ]:
# Cell 9 — Crear JSONL de retry solo para rechazados, EXCLUYENDO Type I

# Esta celda NO lanza batch.
# Solo prepara retry_generation_requests.jsonl para Type II / III / IV.
#
# Type I queda fuera del retry LLM porque ya se regenera localmente
# en la Cell 8.6.

rejected_csv = REPORTS_DIR / "generation_rejected.csv"

if not rejected_csv.exists():
    print("No existe generation_rejected.csv. Corre Cell 8 primero.")
else:
    rejected_df = pd.read_csv(rejected_csv)

    retry_requests = []
    retry_meta = []
    skipped_type1_retry = []
    skipped_non_retryable = []

    RETRYABLE_STATUSES = {
        "rejected_syntax_error",
        "rejected_generation_too_long",
        "rejected_empty_output",
        "rejected_model_error",
    }

    for _, r in rejected_df.iterrows():
        status = str(r.get("status", ""))

        # Algunos errores como line_parse_error pueden venir sin clone_type.
        # No deben romper la celda.
        raw_clone_type = r.get("clone_type", None)

        if pd.isna(raw_clone_type):
            skipped_non_retryable.append({
                "id": r.get("id", ""),
                "status": status,
                "reason": r.get("reason", ""),
                "action": "skipped_missing_clone_type",
            })
            continue

        clone_type = int(raw_clone_type)

        # ============================================================
        # NO mandar Type I al retry LLM
        # ============================================================
        if clone_type == 1:
            skipped_type1_retry.append({
                "id": r.get("id", ""),
                "dataset": r.get("dataset", ""),
                "file_id": r.get("file_id", ""),
                "clone_type": clone_type,
                "status": status,
                "reason": r.get("reason", ""),
                "original_path": r.get("original_path", ""),
                "clone_path": r.get("clone_path", ""),
                "action": "skipped_type1_retry_generated_locally_in_cell_8_6",
            })
            continue

        # ============================================================
        # Solo retry LLM para Type II / III / IV y estados retryables
        # ============================================================
        if status not in RETRYABLE_STATUSES:
            skipped_non_retryable.append({
                "id": r.get("id", ""),
                "dataset": r.get("dataset", ""),
                "file_id": r.get("file_id", ""),
                "clone_type": clone_type,
                "status": status,
                "reason": r.get("reason", ""),
                "action": "skipped_non_retryable_status",
            })
            continue

        dataset_name = r["dataset"]
        file_id = str(r["file_id"])

        original_path = Path(r["original_path"])
        clone_path = Path(r["clone_path"])

        if not original_path.exists():
            skipped_non_retryable.append({
                "id": r.get("id", ""),
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "status": status,
                "reason": "Original path does not exist.",
                "original_path": str(original_path),
                "action": "skipped_missing_original",
            })
            continue

        original_code_raw = read_code(original_path)
        original_code_for_prompt = limit_code_text(original_code_raw)

        custom_id = f"{dataset_name.replace(' ', '_')}|{file_id}_clon_{clone_type}.js"

        repair_note = f"""
Previous generation failed with status: {status}
Failure reason: {str(r.get("reason", ""))[:500]}

Regenerate the clone from scratch.
Do not reuse the broken partial output.
The new output must be a complete syntactically valid JavaScript file.
"""

        prompt_text = (
            CLONE_PROMPTS[clone_type]
            + "\n\nREPAIR MODE:\n"
            + repair_note
            + f"\n\nPair ID: {custom_id}"
            + f"\nDeclared clone type: {clone_type}"
            + f"\nOriginal line count: {len(original_code_raw.splitlines())}"
            + f"\nOriginal character count: {len(original_code_raw)}"
            + "\n\n[ORIGINAL_COMPLETE_FILE_START]\n"
            + original_code_for_prompt
            + "\n[ORIGINAL_COMPLETE_FILE_END]"
        )

        req = {
            "custom_id": custom_id,
            "request": {
                "model": f"models/{MODEL}",
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": prompt_text}],
                    }
                ],
                "generation_config": {
                    "temperature": RETRY_TEMPERATURE_BY_TYPE.get(clone_type, 0.1),
                    "top_p": RETRY_TOP_P_BY_TYPE.get(clone_type, 0.90),
                    "max_output_tokens": RETRY_MAX_OUTPUT_TOKENS_BY_TYPE.get(clone_type, 16384),
                },
            },
        }

        retry_requests.append(req)

        retry_meta.append({
            "id": custom_id,
            "dataset": dataset_name,
            "file_id": file_id,
            "clone_type": clone_type,
            "original_path": str(original_path),
            "clone_path": str(clone_path),
            "previous_status": status,
            "previous_reason": r.get("reason", ""),
        })

    retry_jsonl_path = BATCH_INPUT_DIR / "retry_generation_requests.jsonl"

    with open(retry_jsonl_path, "w", encoding="utf-8") as f:
        for req in retry_requests:
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    with open(REPORTS_DIR / "retry_generation_meta.json", "w", encoding="utf-8") as f:
        json.dump(retry_meta, f, ensure_ascii=False, indent=2)

    skipped_type1_retry_csv = REPORTS_DIR / "skipped_type1_retry.csv"
    skipped_non_retryable_csv = REPORTS_DIR / "skipped_non_retryable_retry.csv"

    pd.DataFrame(skipped_type1_retry).to_csv(skipped_type1_retry_csv, index=False)
    pd.DataFrame(skipped_non_retryable).to_csv(skipped_non_retryable_csv, index=False)

    print("Retry requests Type II/III/IV:", len(retry_requests))
    print("Type I skipped from retry:", len(skipped_type1_retry))
    print("Non-retryable skipped:", len(skipped_non_retryable))
    print("Retry JSONL:", retry_jsonl_path)
    print("Type I skipped CSV:", skipped_type1_retry_csv)
    print("Non-retryable CSV:", skipped_non_retryable_csv)

In [ ]:
# Cell 10A — Lanzar batch de retry para rejected

from pathlib import Path
import json
import time

RETRY_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    retry_jsonl_path
except NameError:
    retry_jsonl_path = BATCH_INPUT_DIR / "retry_generation_requests.jsonl"

retry_jsonl_path = Path(retry_jsonl_path)

if not retry_jsonl_path.exists():
    raise FileNotFoundError(
        f"No existe {retry_jsonl_path}. Primero corre la Cell 9."
    )

retry_request_count = 0
with open(retry_jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            retry_request_count += 1

retry_size_mb = retry_jsonl_path.stat().st_size / (1024 * 1024)

print("=" * 80)
print("RETRY BATCH")
print("=" * 80)
print("Retry JSONL:", retry_jsonl_path)
print(f"Size: {retry_size_mb:.2f} MB")
print("Retry requests:", retry_request_count)

def upload_retry_jsonl_to_gcs(jsonl_path):
    jsonl_path = Path(jsonl_path)

    bucket = storage_client.bucket(BUCKET_NAME)

    blob_name = f"generation/retry_input/run_{RUN_ID}_{RETRY_ID}/{jsonl_path.name}"
    blob = bucket.blob(blob_name)

    # Chunk grande para evitar timeouts en archivos grandes.
    # Debe ser múltiplo de 256 KB.
    blob.chunk_size = 32 * 1024 * 1024

    print("\nUploading retry JSONL to GCS...")
    start = time.time()

    blob.upload_from_filename(
        str(jsonl_path),
        content_type="application/jsonl",
        timeout=3600,
    )

    elapsed = time.time() - start

    gcs_uri = f"gs://{BUCKET_NAME}/{blob_name}"

    print("Upload OK")
    print("GCS URI:", gcs_uri)
    print(f"Upload time: {elapsed/60:.2f} min")

    return gcs_uri


def submit_retry_batch(jsonl_path):
    input_uri = upload_retry_jsonl_to_gcs(jsonl_path)
    output_uri = f"gs://{BUCKET_NAME}/generation/retry_output/run_{RUN_ID}_{RETRY_ID}/"

    print("\nSubmitting retry batch...")
    print("Input :", input_uri)
    print("Output:", output_uri)

    start = time.time()

    job = BatchPredictionJob.submit(
        source_model=MODEL,
        input_dataset=input_uri,
        output_uri_prefix=output_uri,
    )

    elapsed = time.time() - start

    print("\nRetry batch submitted")
    print("Job name:", job.resource_name)
    print("State:", job.state)
    print(f"Submit time: {elapsed:.2f} sec")

    return job.resource_name, output_uri


if retry_request_count == 0:
    print("No hay retry requests. No se lanza batch.")
    retry_job_name = None
    retry_output_uri = None
else:
    retry_job_name, retry_output_uri = submit_retry_batch(retry_jsonl_path)

retry_batch_info = {
    "retry_id": RETRY_ID,
    "retry_job_name": retry_job_name,
    "retry_output_uri": retry_output_uri,
    "retry_jsonl_path": str(retry_jsonl_path),
    "num_retry_requests": retry_request_count,
}

with open(REPORTS_DIR / f"retry_batch_info_{RETRY_ID}.json", "w", encoding="utf-8") as f:
    json.dump(retry_batch_info, f, ensure_ascii=False, indent=2)

retry_batch_info

In [ ]:
# Cell 10 — Reporte final rápido del estado de carpeta

rows = []

for dataset_name, paths in DATASETS.items():
    jsfiles_dir = Path(paths["js_files"])
    clones_dir = Path(paths["clones"])

    for orig_path in sorted(jsfiles_dir.glob("*.js")):
        file_id = orig_path.stem

        if ONLY_FILE_IDS is not None and file_id not in set(map(str, ONLY_FILE_IDS)):
            continue

        original_code = read_code(orig_path) if orig_path.exists() else ""

        for clone_type in CLONE_TYPES_TO_GENERATE:
            clone_path = clones_dir / f"{file_id}_clon_{clone_type}.js"

            row = {
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "original_path": str(orig_path),
                "clone_path": str(clone_path),
                "clone_exists": clone_path.exists(),
                "syntax_ok": None,
                "type1_token_ok": None,
                "status": "",
                "reason": "",
            }

            if not clone_path.exists():
                row["status"] = "missing_clone"
                row["reason"] = "Clone file does not exist"
                rows.append(row)
                continue

            clone_code = read_code(clone_path)

            if not clone_code.strip():
                row["status"] = "empty_clone"
                row["reason"] = "Clone file is empty"
                rows.append(row)
                continue

            syntax_ok, syntax_error = node_syntax_check(clone_path)
            row["syntax_ok"] = syntax_ok

            if not syntax_ok:
                row["status"] = "syntax_error"
                row["reason"] = syntax_error[:500]
                rows.append(row)
                continue

            if clone_type == 1:
                token_ok = type1_tokens_equivalent(original_code, clone_code)
                row["type1_token_ok"] = token_ok
                if not token_ok:
                    row["status"] = "type1_token_mismatch"
                    row["reason"] = "Type I token equivalence failed"
                    rows.append(row)
                    continue

            row["status"] = "local_ok"
            row["reason"] = "Local checks passed"
            rows.append(row)

df_folder = pd.DataFrame(rows)
df_folder.to_csv(REPORTS_DIR / "folder_local_status.csv", index=False)

print("Folder scan rows:", len(df_folder))
display(df_folder["status"].value_counts())
display(pd.crosstab(df_folder["clone_type"], df_folder["status"], margins=True))
print("CSV:", REPORTS_DIR / "folder_local_status.csv")

In [ ]:
# Cell 10B — Revisar estado del retry batch

def refresh_retry_job(retry_job_name):
    if not retry_job_name:
        print("No hay retry_job_name.")
        return None

    job = BatchPredictionJob(retry_job_name)
    job.refresh()

    print("Retry job:", job.resource_name)
    print("State:", job.state)

    return job


try:
    retry_job_name
except NameError:
    # Carga el retry_batch_info más reciente si reiniciaste kernel.
    retry_infos = sorted(REPORTS_DIR.glob("retry_batch_info_*.json"))
    if not retry_infos:
        raise FileNotFoundError("No hay retry_batch_info_*.json. Primero corre la Cell 10A.")
    latest_retry_info = retry_infos[-1]
    print("Cargando:", latest_retry_info)

    with open(latest_retry_info, "r", encoding="utf-8") as f:
        retry_batch_info = json.load(f)

    retry_job_name = retry_batch_info["retry_job_name"]
    retry_output_uri = retry_batch_info["retry_output_uri"]
    RETRY_ID = retry_batch_info["retry_id"]

retry_job = refresh_retry_job(retry_job_name)

In [ ]:
# Cell 10B — Revisar estado del retry batch

def refresh_retry_job(retry_job_name):
    if not retry_job_name:
        print("No hay retry_job_name.")
        return None

    job = BatchPredictionJob(retry_job_name)
    job.refresh()

    print("Retry job:", job.resource_name)
    print("State:", job.state)

    return job


try:
    retry_job_name
except NameError:
    # Carga el retry_batch_info más reciente si reiniciaste kernel.
    retry_infos = sorted(REPORTS_DIR.glob("retry_batch_info_*.json"))
    if not retry_infos:
        raise FileNotFoundError("No hay retry_batch_info_*.json. Primero corre la Cell 10A.")
    latest_retry_info = retry_infos[-1]
    print("Cargando:", latest_retry_info)

    with open(latest_retry_info, "r", encoding="utf-8") as f:
        retry_batch_info = json.load(f)

    retry_job_name = retry_batch_info["retry_job_name"]
    retry_output_uri = retry_batch_info["retry_output_uri"]
    RETRY_ID = retry_batch_info["retry_id"]

retry_job = refresh_retry_job(retry_job_name)

In [ ]:
# Cell 10C — Descargar, parsear y aplicar resultados del retry

retry_rejected_dir = REPORT_DIR / f"retry_rejected_{RETRY_ID}"
retry_rejected_dir.mkdir(parents=True, exist_ok=True)

def download_retry_batch_jsonl(output_uri):
    if not output_uri:
        print("No hay retry output_uri.")
        return []

    bucket = storage_client.bucket(BUCKET_NAME)
    prefix = output_uri.replace(f"gs://{BUCKET_NAME}/", "")

    blobs = list(bucket.list_blobs(prefix=prefix))

    lines = []

    print("Blobs encontrados:", len(blobs))

    for blob in blobs:
        if not blob.name.endswith(".jsonl"):
            continue

        print("Leyendo:", blob.name)
        content = blob.download_as_text()

        for line in content.splitlines():
            if line.strip():
                lines.append(line)

    return lines


def extract_text_from_retry_line(data):
    custom_id = data.get("custom_id") or data.get("id") or "unknown"

    if "error" in data:
        return custom_id, None, f"model_error: {data['error']}"

    try:
        text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
        return custom_id, text, ""
    except Exception as e:
        return custom_id, None, f"parse_error: {e}"


# Cargar metadata del retry
retry_meta_path = REPORTS_DIR / "retry_generation_meta.json"

if not retry_meta_path.exists():
    raise FileNotFoundError(
        f"No existe {retry_meta_path}. Primero corre la Cell 9."
    )

with open(retry_meta_path, "r", encoding="utf-8") as f:
    retry_meta = json.load(f)

retry_meta_by_id = {m["id"]: m for m in retry_meta}

try:
    retry_output_uri
except NameError:
    retry_infos = sorted(REPORTS_DIR.glob("retry_batch_info_*.json"))
    if not retry_infos:
        raise FileNotFoundError("No hay retry_batch_info_*.json. Primero corre la Cell 10A.")
    latest_retry_info = retry_infos[-1]
    print("Cargando:", latest_retry_info)

    with open(latest_retry_info, "r", encoding="utf-8") as f:
        retry_batch_info = json.load(f)

    retry_output_uri = retry_batch_info["retry_output_uri"]
    RETRY_ID = retry_batch_info["retry_id"]

retry_lines = download_retry_batch_jsonl(retry_output_uri)

retry_accepted = []
retry_rejected = []

for raw in retry_lines:
    try:
        data = json.loads(raw)
    except Exception as e:
        retry_rejected.append({
            "id": "unknown",
            "status": "line_parse_error",
            "reason": str(e),
        })
        continue

    custom_id, raw_text, model_error = extract_text_from_retry_line(data)
    m = retry_meta_by_id.get(custom_id, {})

    dataset_name = m.get("dataset", "")
    file_id = str(m.get("file_id", ""))
    clone_type = int(m.get("clone_type", 0)) if m.get("clone_type") is not None else None

    original_path = Path(m.get("original_path", "")) if m.get("original_path") else None
    clone_path = Path(m.get("clone_path", "")) if m.get("clone_path") else None

    row = {
        "id": custom_id,
        "dataset": dataset_name,
        "file_id": file_id,
        "clone_type": clone_type,
        "original_path": str(original_path) if original_path else "",
        "clone_path": str(clone_path) if clone_path else "",
        "status": "",
        "reason": "",
        "syntax_ok": None,
        "type1_token_ok": None,
        "raw_chars": len(raw_text) if raw_text else 0,
        "previous_status": m.get("previous_status", ""),
        "previous_reason": m.get("previous_reason", ""),
    }

    if model_error:
        row["status"] = "retry_rejected_model_error"
        row["reason"] = model_error
        retry_rejected.append(row)
        continue

    code_text = clean_model_code(raw_text)

    if code_text.strip() == "GENERATION_TOO_LONG":
        row["status"] = "retry_rejected_generation_too_long"
        row["reason"] = "Model reported GENERATION_TOO_LONG"
        retry_rejected.append(row)
        continue

    if not code_text.strip():
        row["status"] = "retry_rejected_empty_output"
        row["reason"] = "Empty model output"
        retry_rejected.append(row)
        continue

    temp_path = REPORT_DIR / f"_retry_syntax_{file_id}_clon_{clone_type}.js"
    syntax_ok, syntax_error = syntax_check_text(code_text, temp_path=temp_path)

    row["syntax_ok"] = syntax_ok

    if not syntax_ok:
        row["status"] = "retry_rejected_syntax_error"
        row["reason"] = syntax_error[:500]

        rejected_output_path = retry_rejected_dir / f"{file_id}_clon_{clone_type}.js"
        rejected_output_path.write_text(code_text, encoding="utf-8")
        row["rejected_output_path"] = str(rejected_output_path)

        retry_rejected.append(row)
        continue

    # Type I token check
    if clone_type == 1:
        original_code = read_code(original_path) if original_path and original_path.exists() else ""

        token_ok = type1_tokens_equivalent(original_code, code_text)
        row["type1_token_ok"] = token_ok

        if not token_ok:
            row["status"] = "retry_rejected_type1_token_mismatch"
            row["reason"] = "Type I token-equivalence failed"

            rejected_output_path = retry_rejected_dir / f"{file_id}_clon_{clone_type}.js"
            rejected_output_path.write_text(code_text, encoding="utf-8")
            row["rejected_output_path"] = str(rejected_output_path)

            retry_rejected.append(row)
            continue

    # Guardar clon corregido
    clone_path.parent.mkdir(parents=True, exist_ok=True)
    clone_path.write_text(code_text, encoding="utf-8")

    row["status"] = "retry_accepted"
    row["reason"] = "Retry output passed local checks and was saved."
    retry_accepted.append(row)


retry_all = retry_accepted + retry_rejected

retry_all_df = pd.DataFrame(retry_all)
retry_accepted_df = pd.DataFrame(retry_accepted)
retry_rejected_df = pd.DataFrame(retry_rejected)

retry_all_csv = REPORTS_DIR / f"retry_results_all_{RETRY_ID}.csv"
retry_accepted_csv = REPORTS_DIR / f"retry_accepted_{RETRY_ID}.csv"
retry_rejected_csv = REPORTS_DIR / f"retry_rejected_{RETRY_ID}.csv"

retry_all_df.to_csv(retry_all_csv, index=False)
retry_accepted_df.to_csv(retry_accepted_csv, index=False)
retry_rejected_df.to_csv(retry_rejected_csv, index=False)

print("Retry parse terminado")
print("Retry raw lines:", len(retry_lines))
print("Retry accepted:", len(retry_accepted))
print("Retry rejected:", len(retry_rejected))

print("\nCSVs:")
print("All:", retry_all_csv)
print("Accepted:", retry_accepted_csv)
print("Rejected:", retry_rejected_csv)

if len(retry_all_df):
    display(pd.crosstab(retry_all_df["clone_type"], retry_all_df["status"], margins=True))

if len(retry_rejected_df):
    print("\nRejected reasons:")
    display(retry_rejected_df["status"].value_counts())

In [ ]:
# Cell 10C — Descargar, parsear y aplicar resultados del retry

retry_rejected_dir = REPORT_DIR / f"retry_rejected_{RETRY_ID}"
retry_rejected_dir.mkdir(parents=True, exist_ok=True)

def download_retry_batch_jsonl(output_uri):
    if not output_uri:
        print("No hay retry output_uri.")
        return []

    bucket = storage_client.bucket(BUCKET_NAME)
    prefix = output_uri.replace(f"gs://{BUCKET_NAME}/", "")

    blobs = list(bucket.list_blobs(prefix=prefix))

    lines = []

    print("Blobs encontrados:", len(blobs))

    for blob in blobs:
        if not blob.name.endswith(".jsonl"):
            continue

        print("Leyendo:", blob.name)
        content = blob.download_as_text()

        for line in content.splitlines():
            if line.strip():
                lines.append(line)

    return lines


def extract_text_from_retry_line(data):
    custom_id = data.get("custom_id") or data.get("id") or "unknown"

    if "error" in data:
        return custom_id, None, f"model_error: {data['error']}"

    try:
        text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
        return custom_id, text, ""
    except Exception as e:
        return custom_id, None, f"parse_error: {e}"


# Cargar metadata del retry
retry_meta_path = REPORTS_DIR / "retry_generation_meta.json"

if not retry_meta_path.exists():
    raise FileNotFoundError(
        f"No existe {retry_meta_path}. Primero corre la Cell 9."
    )

with open(retry_meta_path, "r", encoding="utf-8") as f:
    retry_meta = json.load(f)

retry_meta_by_id = {m["id"]: m for m in retry_meta}

try:
    retry_output_uri
except NameError:
    retry_infos = sorted(REPORTS_DIR.glob("retry_batch_info_*.json"))
    if not retry_infos:
        raise FileNotFoundError("No hay retry_batch_info_*.json. Primero corre la Cell 10A.")
    latest_retry_info = retry_infos[-1]
    print("Cargando:", latest_retry_info)

    with open(latest_retry_info, "r", encoding="utf-8") as f:
        retry_batch_info = json.load(f)

    retry_output_uri = retry_batch_info["retry_output_uri"]
    RETRY_ID = retry_batch_info["retry_id"]

retry_lines = download_retry_batch_jsonl(retry_output_uri)

retry_accepted = []
retry_rejected = []

for raw in retry_lines:
    try:
        data = json.loads(raw)
    except Exception as e:
        retry_rejected.append({
            "id": "unknown",
            "status": "line_parse_error",
            "reason": str(e),
        })
        continue

    custom_id, raw_text, model_error = extract_text_from_retry_line(data)
    m = retry_meta_by_id.get(custom_id, {})

    dataset_name = m.get("dataset", "")
    file_id = str(m.get("file_id", ""))
    clone_type = int(m.get("clone_type", 0)) if m.get("clone_type") is not None else None

    original_path = Path(m.get("original_path", "")) if m.get("original_path") else None
    clone_path = Path(m.get("clone_path", "")) if m.get("clone_path") else None

    row = {
        "id": custom_id,
        "dataset": dataset_name,
        "file_id": file_id,
        "clone_type": clone_type,
        "original_path": str(original_path) if original_path else "",
        "clone_path": str(clone_path) if clone_path else "",
        "status": "",
        "reason": "",
        "syntax_ok": None,
        "type1_token_ok": None,
        "raw_chars": len(raw_text) if raw_text else 0,
        "previous_status": m.get("previous_status", ""),
        "previous_reason": m.get("previous_reason", ""),
    }

    if model_error:
        row["status"] = "retry_rejected_model_error"
        row["reason"] = model_error
        retry_rejected.append(row)
        continue

    code_text = clean_model_code(raw_text)

    if code_text.strip() == "GENERATION_TOO_LONG":
        row["status"] = "retry_rejected_generation_too_long"
        row["reason"] = "Model reported GENERATION_TOO_LONG"
        retry_rejected.append(row)
        continue

    if not code_text.strip():
        row["status"] = "retry_rejected_empty_output"
        row["reason"] = "Empty model output"
        retry_rejected.append(row)
        continue

    temp_path = REPORT_DIR / f"_retry_syntax_{file_id}_clon_{clone_type}.js"
    syntax_ok, syntax_error = syntax_check_text(code_text, temp_path=temp_path)

    row["syntax_ok"] = syntax_ok

    if not syntax_ok:
        row["status"] = "retry_rejected_syntax_error"
        row["reason"] = syntax_error[:500]

        rejected_output_path = retry_rejected_dir / f"{file_id}_clon_{clone_type}.js"
        rejected_output_path.write_text(code_text, encoding="utf-8")
        row["rejected_output_path"] = str(rejected_output_path)

        retry_rejected.append(row)
        continue

    # Type I token check
    if clone_type == 1:
        original_code = read_code(original_path) if original_path and original_path.exists() else ""

        token_ok = type1_tokens_equivalent(original_code, code_text)
        row["type1_token_ok"] = token_ok

        if not token_ok:
            row["status"] = "retry_rejected_type1_token_mismatch"
            row["reason"] = "Type I token-equivalence failed"

            rejected_output_path = retry_rejected_dir / f"{file_id}_clon_{clone_type}.js"
            rejected_output_path.write_text(code_text, encoding="utf-8")
            row["rejected_output_path"] = str(rejected_output_path)

            retry_rejected.append(row)
            continue

    # Guardar clon corregido
    clone_path.parent.mkdir(parents=True, exist_ok=True)
    clone_path.write_text(code_text, encoding="utf-8")

    row["status"] = "retry_accepted"
    row["reason"] = "Retry output passed local checks and was saved."
    retry_accepted.append(row)


retry_all = retry_accepted + retry_rejected

retry_all_df = pd.DataFrame(retry_all)
retry_accepted_df = pd.DataFrame(retry_accepted)
retry_rejected_df = pd.DataFrame(retry_rejected)

retry_all_csv = REPORTS_DIR / f"retry_results_all_{RETRY_ID}.csv"
retry_accepted_csv = REPORTS_DIR / f"retry_accepted_{RETRY_ID}.csv"
retry_rejected_csv = REPORTS_DIR / f"retry_rejected_{RETRY_ID}.csv"

retry_all_df.to_csv(retry_all_csv, index=False)
retry_accepted_df.to_csv(retry_accepted_csv, index=False)
retry_rejected_df.to_csv(retry_rejected_csv, index=False)

print("Retry parse terminado")
print("Retry raw lines:", len(retry_lines))
print("Retry accepted:", len(retry_accepted))
print("Retry rejected:", len(retry_rejected))

print("\nCSVs:")
print("All:", retry_all_csv)
print("Accepted:", retry_accepted_csv)
print("Rejected:", retry_rejected_csv)

if len(retry_all_df):
    display(pd.crosstab(retry_all_df["clone_type"], retry_all_df["status"], margins=True))

if len(retry_rejected_df):
    print("\nRejected reasons:")
    display(retry_rejected_df["status"].value_counts())